In [1]:
# 1
import cadquery as cq
from jupyter_cadquery import show

c = cq.Workplane('front')
c = c.box(1, 2, 2).faces('>X').chamfer(0.1)

c

Overwriting auto display for cadquery Workplane and Shape
+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [2]:
#2
import cadquery as cq

cq.Workplane(cq.Plane.XY()).box(4, 2, 0.5).faces(">Z") \
             .workplane().rect(3.5, 1.5, forConstruction=True) \
             .vertices().cskHole(0.125, 0.3, 82.0, depth=None)
#parameter definitions
p_outerWidth = 100.0 #Outer width of box enclosure
p_outerLength = 150.0 #Outer length of box enclosure
p_outerHeight = 50.0 #Outer height of box enclosure

p_thickness =  3.0 #Thickness of the box walls
p_sideRadius =  10.0 #Radius for the curves around the sides of the bo
p_topAndBottomRadius =  2.0 #Radius for the curves on the top and bottom edges of the box

p_screwpostInset = 12.0 #How far in from the edges the screwposts should be place.
p_screwpostID = 4.0 #nner Diameter of the screwpost holes, should be roughly screw diameter not including threads
p_screwpostOD = 10.0 #Outer Diameter of the screwposts.\nDetermines overall thickness of the posts

p_boreDiameter = 8.0 #Diameter of the counterbore hole, if any
p_boreDepth = 1.0 #Depth of the counterbore hole, if
p_countersinkDiameter = 0.0 #Outer diameter of countersink.  Should roughly match the outer diameter of the screw head
p_countersinkAngle = 90.0 #Countersink angle (complete angle between opposite sides, not from center to one side)
p_flipLid = True #Whether to place the lid with the top facing down or not.
p_lipHeight =  1.0 #Height of lip on the underside of the lid.\nSits inside the box body for a snug fit.

#outer shell
oshell = cq.Workplane("XY").rect(p_outerWidth,p_outerLength).extrude(p_outerHeight + p_lipHeight)

#weird geometry happens if we make the fillets in the wrong order
if p_sideRadius > p_topAndBottomRadius:
    oshell.edges("|Z").fillet(p_sideRadius)
    oshell.edges("#Z").fillet(p_topAndBottomRadius)
else:
    oshell.edges("#Z").fillet(p_topAndBottomRadius)
    oshell.edges("|Z").fillet(p_sideRadius)

#inner shell
ishell = oshell.faces("<Z").workplane(p_thickness,True)\
    .rect((p_outerWidth - 2.0* p_thickness),(p_outerLength - 2.0*p_thickness))\
    .extrude((p_outerHeight - 2.0*p_thickness),False) #set combine false to produce just the new boss
ishell.edges("|Z").fillet(p_sideRadius - p_thickness)

#make the box outer box
box = oshell.cut(ishell)

#make the screwposts
POSTWIDTH = (p_outerWidth - 2.0*p_screwpostInset)
POSTLENGTH = (p_outerLength  -2.0*p_screwpostInset)

postCenters = box.faces(">Z").workplane(-p_thickness)\
    .rect(POSTWIDTH,POSTLENGTH,forConstruction=True)\
    .vertices()

for v in postCenters.all():
   v.circle(p_screwpostOD/2.0).circle(p_screwpostID/2.0)\
        .extrude((-1.0)*(p_outerHeight + p_lipHeight -p_thickness ),True)

#split lid into top and bottom parts
(lid,bottom) = box.faces(">Z").workplane(-p_thickness -p_lipHeight ).split(keepTop=True,keepBottom=True).all()  #splits into two solids

#translate the lid, and subtract the bottom from it to produce the lid inset
lowerLid = lid.translate((0,0,-p_lipHeight))
cutlip = lowerLid.cut(bottom).translate((p_outerWidth + p_thickness ,0,p_thickness - p_outerHeight + p_lipHeight))

#compute centers for counterbore/countersink or counterbore
topOfLidCenters = cutlip.faces(">Z").workplane().rect(POSTWIDTH,POSTLENGTH,forConstruction=True).vertices()

#add holes of the desired type
if p_boreDiameter > 0 and p_boreDepth > 0:
    topOfLid = topOfLidCenters.cboreHole(p_screwpostID,p_boreDiameter,p_boreDepth,(2.0)*p_thickness)
elif p_countersinkDiameter > 0 and p_countersinkAngle > 0:
    topOfLid = topOfLidCenters.cskHole(p_screwpostID,p_countersinkDiameter,p_countersinkAngle,(2.0)*p_thickness)
else:
    topOfLid= topOfLidCenters.hole(p_screwpostID,(2.0)*p_thickness)

#flip lid upside down if desired
if p_flipLid:
    topOfLid.rotateAboutCenter((1,0,0),180)

result =topOfLid.combineSolids(bottom)
    
#return the combined result
show(result)

+


/Users/kozy/miniforge3/envs/cadquery/lib/python3.13/site-packages/cadquery/utils.py:39: FutureWarning: combineSolids will be removed in the next release.
  warn(f"{f.__name__} will be removed in the next release.", FutureWarning)


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [1]:
# 3
import cadquery as cq
from jupyter_cadquery import show
#####
# Inputs
######
lbumps = 6       # number of bumps long
wbumps = 2       # number of bumps wide
thin = True      # True for thin, False for thick

#
# Lego Brick Constants-- these make a lego brick a lego :)
#
pitch = 8.0
clearance = 0.1
bumpDiam = 4.8
bumpHeight = 1.8
if thin:
    height = 3.2
else:
    height = 9.6

t = (pitch - (2 * clearance) - bumpDiam) / 2.0
postDiam = pitch - t  # works out to 6.5
total_length = lbumps*pitch - 2.0*clearance
total_width = wbumps*pitch - 2.0*clearance

# make the base
s = cq.Workplane("XY").box(total_length, total_width, height)

# shell inwards not outwards
s = s.faces("<Z").shell(-1.0 * t)

# make the bumps on the top
s = s.faces(">Z").workplane(). \
    rarray(pitch, pitch, lbumps, wbumps, True).circle(bumpDiam / 2.0) \
    .extrude(bumpHeight)

# add posts on the bottom. posts are different diameter depending on geometry
# solid studs for 1 bump, tubes for multiple, none for 1x1
tmp = s.faces("<Z").workplane(invert=True)

if lbumps > 1 and wbumps > 1:
    tmp = tmp.rarray(pitch, pitch, lbumps - 1, wbumps - 1, center=True). \
        circle(postDiam / 2.0).circle(bumpDiam / 2.0).extrude(height - t)
elif lbumps > 1:
    tmp = tmp.rarray(pitch, pitch, lbumps - 1, 1, center=True). \
        circle(t).extrude(height - t)
elif wbumps > 1:
    tmp = tmp.rarray(pitch, pitch, 1, wbumps - 1, center=True). \
        circle(t).extrude(height - t)
else:
    tmp = s
s

Overwriting auto display for cadquery Workplane and Shape
+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [2]:
# 4
import cadquery as cq
from jupyter_cadquery import show
(L,w,t) = (20.0, 6.0, 3.0)
s = cq.Workplane("XY")

#draw half the profile of the bottle and extrude it
p = s.center(-L/2.0, 0).vLine(w/2.0) \
    .threePointArc((L/2.0, w/2.0 + t),(L, w/2.0)).vLine(-w/2.0) \
    .mirrorX().extrude(30.0,True)

#make the neck
p.faces(">Z").workplane().circle(3.0).extrude(2.0,True)

#make a shell
result = p.faces(">Z").shell(0.3)

result

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [3]:
#5
import cadquery as cq

result = cq.Workplane("XY").box(2, 2, 2).\
    faces(">Z").shell(-0.2).\
    faces(">Z").edges("not(<X or >X or <Y or >Y)").\
    chamfer(0.125, 0.02)
    
result

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [7]:
#6
import cadquery as cq
from cqterrain import damage 

blast_ex = damage.blast(
    seed="test",
    height=10,
    count = (5,10),
    x_jiggle = (-2,2), 
    y_jiggle = 0,
    ring_params = [
        {"radius":(35,50), "start_angle":0}, 
        {"radius":25,"start_angle":30}
    ]
)



show(blast_ex)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [8]:
#7
import cadquery as cq
from cqterrain.book import Bookcase

bp_case = Bookcase()
bp_case.length = 100
bp_case.width = 15
bp_case.segments = 4
bp_case.minus_width = 3
bp_case.seed = "purple"
bp_case.book_count =(16,30,1)
bp_case.min_book_height = 6

#closed
bp_case.bottom_align = True
bp_case.page_width_inset=0.5
bp_case.back_translate = 1

# open
#bp_case.bottom_align = False#True
#bp_case.page_width_inset=1#0.5
#bp_case.back_translate = 0#1

bp_case.render_books = True
bp_case.make()

ex_case = bp_case.build()
show(ex_case)

cq.exporters.export(ex_case,'stl/book_bookcase_books.stl')

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [9]:
#8
import cadquery as cq
from jupyter_cadquery import show

# Book parameters
length = 3
width = 10
height = 12
binder_width = 0.5
page_width = 8
page_height = 11
fillet_size = 0.5

# Create the book
cover = cq.Workplane("XY").box(length, width, height)
interior = cq.Workplane("XY").box(length-binder_width*2, width-binder_width, height)
pages = cq.Workplane("XY").box(length-binder_width*2, page_width, page_height)

page_offset = (width - page_width)/2

cover = cover.faces("-Y").edges("Z").fillet(fillet_size)

book_model = (
    cq.Workplane("XY")
    .union(cover)
    .cut(interior.translate((0, binder_width/2, 0)))
    .union(pages.translate((0, -page_offset+binder_width, 0)))
)

show(book_model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [2]:
#9
import cadquery as cq
from jupyter_cadquery import show

# Building parameters
length = 100
width = 80
height = 120
wall_thickness = 20

# Window parameters
window_width = 15
window_height = 20
window_count = 4

# Create building shell
building = cq.Workplane("XY").box(length, width, height)
interior = cq.Workplane("XY").box(length - wall_thickness*2, width - wall_thickness*2, height - wall_thickness)
building = building.cut(interior)

# Add windows on one wall
window_spacing = length / (window_count + 1)
for i in range(window_count):
    x_pos = -length/2 + window_spacing * (i + 1)
    window = cq.Workplane("XY").box(window_width, wall_thickness + 1, window_height)
    window = window.translate((x_pos, width/2, 0))
    building = building.cut(window)

show(building)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [3]:
#10
import cadquery as cq
from jupyter_cadquery import show
import math

# Impeller parameters
hub_diameter = 30
hub_height = 20
shroud_diameter = 100
blade_count = 6
blade_thickness = 3
blade_height = 25
inlet_diameter = 25

# Create hub (center shaft mounting)
hub = cq.Workplane("XY").cylinder(hub_height, hub_diameter/2)

# Create shaft bore with keyway
bore_diameter = 12
keyway_width = 4
keyway_depth = 2
bore = cq.Workplane("XY").cylinder(hub_height + 1, bore_diameter/2)
keyway = cq.Workplane("XY").box(keyway_width, bore_diameter, hub_height + 1)
hub = hub.cut(bore).cut(keyway)

# Create backplate (bottom disk)
backplate = cq.Workplane("XY").cylinder(blade_thickness, shroud_diameter/2)
backplate = backplate.translate((0, 0, -blade_thickness/2))

# Create shroud (top disk with inlet hole)
shroud = cq.Workplane("XY").cylinder(blade_thickness, shroud_diameter/2)
shroud = shroud.translate((0, 0, hub_height + blade_height - blade_thickness/2))
inlet_hole = cq.Workplane("XY").cylinder(blade_thickness + 1, inlet_diameter/2)
inlet_hole = inlet_hole.translate((0, 0, hub_height + blade_height - blade_thickness/2))
shroud = shroud.cut(inlet_hole)

# Combine base components
impeller = cq.Workplane("XY").union(hub).union(backplate).union(shroud)

# Create curved impeller blades
for i in range(blade_count):
    angle = 360 / blade_count * i
    
    # Define blade profile points (curved backward-swept blade)
    points = []
    num_points = 20
    for j in range(num_points):
        t = j / (num_points - 1)
        
        # Radial position (from hub to shroud)
        r = hub_diameter/2 + t * (shroud_diameter/2 - hub_diameter/2 - blade_thickness)
        
        # Tangential sweep angle (backward-curved)
        sweep_angle = -45 * t  # Backward sweep
        theta = math.radians(angle + sweep_angle)
        
        # Height varies from bottom to top
        z = t * blade_height
        
        x = r * math.cos(theta)
        y = r * math.sin(theta)
        points.append((x, y, z))
    
    # Create blade surface using lofted spline
    blade_path = cq.Workplane("XY")
    for idx, (x, y, z) in enumerate(points):
        if idx == 0:
            blade_path = blade_path.moveTo(x, y)
        else:
            # Create intermediate profiles
            blade_section = cq.Workplane("XY").center(x, y).rect(blade_thickness, blade_thickness)
            blade_section = blade_section.extrude(0.1).translate((0, 0, z))
    
    # Create blade using sweep
    # Simplified: use extruded spline with rotation
    blade_points_2d = [(p[0], p[1]) for p in points]
    
    # Create blade as a swept solid
    blade_base = cq.Workplane("XY").moveTo(points[0][0], points[0][1])
    for p in points[1:]:
        blade_base = blade_base.lineTo(p[0], p[1])
    
    blade = blade_base.close().extrude(blade_height)
    
    # Create thickness by offsetting
    blade_inner = cq.Workplane("XY").moveTo(points[0][0] * 0.95, points[0][1] * 0.95)
    for p in points[1:]:
        blade_inner = blade_inner.lineTo(p[0] * 0.95, p[1] * 0.95)
    blade_inner = blade_inner.close().extrude(blade_height)
    
    # Alternative: Create blade using box and transformations
    # Simplified blade as twisted, tapered shape
    for j in range(len(points) - 1):
        x1, y1, z1 = points[j]
        x2, y2, z2 = points[j + 1]
        
        # Calculate blade segment
        dx = x2 - x1
        dy = y2 - y1
        dz = z2 - z1
        length = math.sqrt(dx**2 + dy**2)
        
        if length > 0:
            segment_angle = math.degrees(math.atan2(dy, dx))
            
            blade_segment = cq.Workplane("XY").box(length, blade_thickness, dz + 0.5)
            blade_segment = blade_segment.translate((length/2, 0, 0))
            blade_segment = blade_segment.rotate((0, 0, 0), (0, 0, 1), segment_angle)
            blade_segment = blade_segment.translate((x1, y1, z1 + dz/2))
            
            impeller = impeller.union(blade_segment)

# Add mounting holes on backplate
mounting_holes_diameter = 8
mounting_holes_circle_diameter = shroud_diameter * 0.8
num_mounting_holes = 4

for i in range(num_mounting_holes):
    angle = 360 / num_mounting_holes * i
    x = (mounting_holes_circle_diameter/2) * math.cos(math.radians(angle))
    y = (mounting_holes_circle_diameter/2) * math.sin(math.radians(angle))
    
    hole = cq.Workplane("XY").cylinder(blade_thickness + 1, mounting_holes_diameter/2)
    hole = hole.translate((x, y, -blade_thickness/2))
    impeller = impeller.cut(hole)

# Add balancing holes (weight reduction)
balance_hole_diameter = 6
balance_holes_circle_diameter = hub_diameter * 1.8
num_balance_holes = 3

for i in range(num_balance_holes):
    angle = 360 / num_balance_holes * i + 60  # Offset from blades
    x = (balance_holes_circle_diameter/2) * math.cos(math.radians(angle))
    y = (balance_holes_circle_diameter/2) * math.sin(math.radians(angle))
    
    hole = cq.Workplane("XY").cylinder(hub_height, balance_hole_diameter/2)
    hole = hole.translate((x, y, hub_height/2))
    impeller = impeller.cut(hole)

# Add chamfers for manufacturing
impeller = impeller.faces(">Z").edges().chamfer(1)
impeller = impeller.faces("<Z").edges().chamfer(0.5)

show(impeller)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [14]:
#11
import cadquery as cq
from jupyter_cadquery import show
import math

# Ornate Chess Rook - Castle tower chess piece with detailed features
# Parameters
base_diameter = 30.0        # Base of the rook
base_height = 8.0           # Height of base platform
body_bottom_dia = 22.0      # Bottom diameter of main body
body_top_dia = 18.0         # Top diameter of main body
body_height = 35.0          # Height of main body section
neck_diameter = 14.0        # Diameter of neck section
neck_height = 8.0           # Height of neck

# Battlement parameters
battlement_outer_dia = 24.0  # Outer diameter of battlements
battlement_inner_dia = 16.0  # Inner diameter (hollow center)
battlement_height = 15.0     # Height of battlement section
merlon_width = 5.0           # Width of each merlon (raised section)
num_merlons = 8              # Number of merlons around the top
merlon_height = 8.0          # Height of merlons above base

# Decorative band parameters
band_positions = [15.0, 25.0, 35.0]  # Y positions for decorative bands
band_height = 2.0            # Height of each band
band_depth = 1.0             # Depth of band grooves

# Window parameters
window_height = 8.0          # Height of castle windows
window_width = 3.0           # Width of castle windows
window_y_pos = 20.0          # Vertical position of windows
num_windows = 4              # Number of windows around body

# Base spiral groove parameters
spiral_turns = 2.5           # Number of spiral turns
spiral_depth = 0.8           # Depth of spiral groove
spiral_width = 1.5           # Width of spiral groove

# Create the base platform with beveled edge
model = (cq.Workplane("XY")
    .circle(base_diameter / 2)
    .extrude(base_height / 2)
)

# Add upper base section (slightly smaller)
upper_base = (cq.Workplane("XY")
    .workplane(offset=base_height / 2)
    .circle(base_diameter / 2 - 2)
    .extrude(base_height / 2)
)
model = model.union(upper_base)

# Create main body with taper using loft
body = (cq.Workplane("XY")
    .workplane(offset=base_height)
    .circle(body_bottom_dia / 2)
    .workplane(offset=body_height)
    .circle(body_top_dia / 2)
    .loft()
)
model = model.union(body)

# Add neck section
neck = (cq.Workplane("XY")
    .workplane(offset=base_height + body_height)
    .circle(neck_diameter / 2)
    .extrude(neck_height)
)
model = model.union(neck)

# Create battlement base
battlement_base = (cq.Workplane("XY")
    .workplane(offset=base_height + body_height + neck_height)
    .circle(battlement_outer_dia / 2)
    .extrude(battlement_height - merlon_height)
)
model = model.union(battlement_base)

# Cut hollow center through battlements
hollow_cut = (cq.Workplane("XY")
    .workplane(offset=base_height + body_height + neck_height)
    .circle(battlement_inner_dia / 2)
    .extrude(battlement_height + 1)
)
model = model.cut(hollow_cut)

# Add merlons (raised sections) using trigonometric positioning
merlon_angle = 2 * math.pi / num_merlons
merlon_arc_angle = merlon_angle * 0.4  # Merlons take up 40% of circumference

for i in range(num_merlons):
    angle = i * merlon_angle
    
    # Calculate merlon position
    center_x = (battlement_outer_dia / 2 + battlement_inner_dia / 2) / 2 * math.cos(angle)
    center_y = (battlement_outer_dia / 2 + battlement_inner_dia / 2) / 2 * math.sin(angle)
    
    # Create merlon as a small box rotated to match position
    merlon = (cq.Workplane("XY")
        .workplane(offset=base_height + body_height + neck_height + battlement_height - merlon_height)
        .center(center_x, center_y)
        .box(merlon_width, (battlement_outer_dia - battlement_inner_dia) / 2, merlon_height,
             centered=(True, True, False))
        .rotate((0, 0, 0), (0, 0, 1), math.degrees(angle))
    )
    model = model.union(merlon)

# Add decorative bands around the body using loops
for band_y in band_positions:
    if band_y < body_height:  # Only add bands within body height
        # Calculate radius at this height (linear interpolation for taper)
        t = band_y / body_height
        band_radius = body_bottom_dia / 2 * (1 - t) + body_top_dia / 2 * t
        
        # Create band groove
        band_cut = (cq.Workplane("XY")
            .workplane(offset=base_height + band_y)
            .circle(band_radius)
            .circle(band_radius - band_depth)
            .extrude(band_height)
        )
        model = model.cut(band_cut)

# Add castle windows using parametric array
for i in range(num_windows):
    angle = 2 * math.pi * i / num_windows
    
    # Calculate window position on body surface
    window_radius = (body_bottom_dia / 2 + body_top_dia / 2) / 2
    x_pos = window_radius * math.cos(angle)
    y_pos = window_radius * math.sin(angle)
    
    # Create arched window cutout
    window_cut = (cq.Workplane("XZ")
        .workplane(offset=y_pos)
        .center(x_pos, base_height + window_y_pos)
        .rect(window_width, window_height)
        .workplane(offset=-window_radius)
        .rect(window_width, window_height)
        .loft()
    )
    
    # Add arch top to window
    arch_cut = (cq.Workplane("XZ")
        .workplane(offset=y_pos)
        .center(x_pos, base_height + window_y_pos + window_height/2)
        .circle(window_width/2)
        .workplane(offset=-window_radius)
        .circle(window_width/2)
        .loft()
    )
    window_cut = window_cut.union(arch_cut)
    model = model.cut(window_cut)

# Add spiral groove around base using mathematical helix
num_points = 100  # Points for spiral path
points = []
for i in range(num_points):
    t = i / (num_points - 1)
    angle = t * spiral_turns * 2 * math.pi
    radius = base_diameter / 2 - 1
    x = radius * math.cos(angle)
    y = radius * math.sin(angle)
    z = 1 + t * (base_height - 2)
    points.append((x, y, z))

# Create spiral groove by cutting small cylinders along path
for i in range(0, len(points) - 1, 5):  # Sample every 5th point for efficiency
    point = points[i]
    next_point = points[i + 1] if i + 1 < len(points) else points[i]
    
    # Calculate groove direction
    dx = next_point[0] - point[0]
    dy = next_point[1] - point[1]
    length = math.sqrt(dx*dx + dy*dy) if dx != 0 or dy != 0 else 0.1
    
    if length > 0:
        groove_cut = (cq.Workplane("XY")
            .workplane(offset=point[2])
            .center(point[0], point[1])
            .circle(spiral_width / 2)
            .extrude(spiral_depth * 2)
        )
        model = model.cut(groove_cut)

# Add cross-shaped slot on top for authentic rook design
cross_length = battlement_inner_dia - 2
cross_width = 3.0
cross_depth = 5.0
top_z = base_height + body_height + neck_height + battlement_height - merlon_height - cross_depth

# Vertical part of cross
cross_v = (cq.Workplane("XY")
    .workplane(offset=top_z)
    .box(cross_width, cross_length, cross_depth,
         centered=(True, True, False))
)

# Horizontal part of cross
cross_h = (cq.Workplane("XY")
    .workplane(offset=top_z)
    .box(cross_length, cross_width, cross_depth,
         centered=(True, True, False))
)

cross_cut = cross_v.union(cross_h)
model = model.cut(cross_cut)

# Add small detail dots around neck using loops
dot_radius = 0.5
num_dots = 12
for i in range(num_dots):
    angle = 2 * math.pi * i / num_dots
    x = neck_diameter / 2 * math.cos(angle)
    y = neck_diameter / 2 * math.sin(angle)
    
    dot_cut = (cq.Workplane("XY")
        .workplane(offset=base_height + body_height + neck_height / 2)
        .center(x, y)
        .circle(dot_radius)
        .extrude(dot_radius)
    )
    model = model.cut(dot_cut)

show(model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [15]:
#12
import cadquery as cq
from jupyter_cadquery import show
import math

# Elegant Spinning Top - Precision-balanced toy with decorative spiral pattern
# Parameters
base_radius = 25.0          # Widest point radius
tip_radius = 1.5            # Tip point radius
total_height = 60.0         # Total height of top
body_height = 45.0          # Height of main body
handle_radius = 4.0         # Handle stem radius
handle_height = 12.0        # Handle height

# Spiral groove parameters
num_spirals = 3             # Number of spiral grooves
spiral_turns = 2.5          # Rotations of spiral
groove_depth = 1.2          # Depth of grooves
groove_width = 2.0          # Width of grooves

# Ring decoration parameters
ring_positions = [15, 25, 35]  # Y-positions for decorative rings
ring_depth = 0.8              # Depth of ring grooves

# Create main body using lofted profiles for smooth shape
# Bottom point
profile1 = (cq.Workplane("XY")
    .circle(tip_radius)
)

# Widest point (1/3 up)
profile2 = (cq.Workplane("XY")
    .workplane(offset=body_height * 0.35)
    .circle(base_radius)
)

# Middle narrow point (2/3 up)
profile3 = (cq.Workplane("XY")
    .workplane(offset=body_height * 0.7)
    .circle(base_radius * 0.4)
)

# Top of body
profile4 = (cq.Workplane("XY")
    .workplane(offset=body_height)
    .circle(handle_radius * 1.5)
)

# Create main body by lofting through profiles
model = (cq.Workplane("XY")
    .circle(tip_radius)
    .workplane(offset=body_height * 0.35)
    .circle(base_radius)
    .workplane(offset=body_height * 0.35)
    .circle(base_radius * 0.4)
    .workplane(offset=body_height * 0.3)
    .circle(handle_radius * 1.5)
    .loft(combine=True)
)

# Add handle stem with taper
handle = (cq.Workplane("XY")
    .workplane(offset=body_height)
    .circle(handle_radius * 1.2)
    .workplane(offset=handle_height * 0.5)
    .circle(handle_radius)
    .workplane(offset=handle_height * 0.5)
    .circle(handle_radius * 1.1)
    .loft(combine=True)
)
model = model.union(handle)

# Cut spiral grooves using mathematical helix
for spiral_idx in range(num_spirals):
    phase = 2 * math.pi * spiral_idx / num_spirals
    points_per_spiral = 60
    
    for i in range(points_per_spiral):
        t = i / points_per_spiral
        angle = phase + t * spiral_turns * 2 * math.pi
        
        # Calculate radius at this height (following body profile)
        height = t * body_height * 0.7  # Spirals go up to 70% of body
        if height < body_height * 0.35:
            r = tip_radius + (base_radius - tip_radius) * (height / (body_height * 0.35))
        else:
            r = base_radius * (1 - 0.6 * ((height - body_height * 0.35) / (body_height * 0.35)))
        
        x = r * math.cos(angle)
        y = r * math.sin(angle)
        
        # Create groove cut
        groove = (cq.Workplane("XY")
            .workplane(offset=height)
            .center(x, y)
            .circle(groove_width / 2)
            .extrude(groove_depth * 2)
        )
        model = model.cut(groove)

# Add decorative rings at specified heights
for ring_y in ring_positions:
    if ring_y < body_height * 0.7:  # Only add rings within spiral area
        # Calculate radius at this height
        if ring_y < body_height * 0.35:
            radius = tip_radius + (base_radius - tip_radius) * (ring_y / (body_height * 0.35))
        else:
            radius = base_radius * (1 - 0.6 * ((ring_y - body_height * 0.35) / (body_height * 0.35)))
        
        ring_cut = (cq.Workplane("XY")
            .workplane(offset=ring_y)
            .circle(radius)
            .circle(radius - ring_depth)
            .extrude(1.5)
        )
        model = model.cut(ring_cut)

# Add grip pattern to handle - vertical flutes
num_flutes = 8
for i in range(num_flutes):
    angle = 2 * math.pi * i / num_flutes
    flute_x = (handle_radius + 0.5) * math.cos(angle)
    flute_y = (handle_radius + 0.5) * math.sin(angle)
    
    flute = (cq.Workplane("XY")
        .workplane(offset=body_height + 2)
        .center(flute_x, flute_y)
        .circle(0.8)
        .extrude(handle_height - 4)
    )
    model = model.cut(flute)

# Add small divot at very tip for better spinning
tip_divot = (cq.Workplane("XY")
    .sphere(tip_radius * 0.8)
)
model = model.cut(tip_divot)

show(model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [4]:
#13 
import cadquery as cq
from jupyter_cadquery import show
import math

# Art Deco Perfume Bottle - Geometric elegance with faceted design
# Parameters
base_width = 40.0           # Base rectangle width
base_depth = 25.0           # Base rectangle depth
bottle_height = 55.0        # Main bottle height
neck_height = 15.0          # Neck height
cap_height = 12.0           # Stopper cap height

# Facet parameters
num_facets = 8              # Number of vertical facets
facet_taper = 0.7           # Taper ratio for top vs bottom

# Pattern parameters
diamond_size = 4.0          # Size of diamond pattern cuts
pattern_depth = 1.0         # Depth of surface patterns
pattern_rows = 6            # Number of pattern rows

# Neck and stopper parameters
neck_radius = 6.0           # Neck bottle radius
stopper_base = 10.0         # Stopper base width
stopper_top = 12.0          # Stopper top width

# Create main bottle body with octagonal facets
# Bottom profile - rectangular with chamfered corners
base_points = []
corner_cut = 8.0
base_points = [
    (-base_width/2 + corner_cut, -base_depth/2),
    (base_width/2 - corner_cut, -base_depth/2),
    (base_width/2, -base_depth/2 + corner_cut),
    (base_width/2, base_depth/2 - corner_cut),
    (base_width/2 - corner_cut, base_depth/2),
    (-base_width/2 + corner_cut, base_depth/2),
    (-base_width/2, base_depth/2 - corner_cut),
    (-base_width/2, -base_depth/2 + corner_cut)
]

# Create bottle body with taper
model = (cq.Workplane("XY")
    .polyline(base_points)
    .close()
    .workplane(offset=bottle_height)
    .polygon(num_facets, base_width * facet_taper)
    .loft(combine=True)
)

# Add elegant neck transition
neck_transition = (cq.Workplane("XY")
    .workplane(offset=bottle_height)
    .polygon(num_facets, base_width * facet_taper)
    .workplane(offset=8)
    .circle(neck_radius + 3)
    .loft(combine=True)
)
model = model.union(neck_transition)

# Add bottle neck
neck = (cq.Workplane("XY")
    .workplane(offset=bottle_height + 8)
    .circle(neck_radius)
    .extrude(neck_height - 8)
)
model = model.union(neck)

# Create diamond pattern on front and back faces
for face_y in [-base_depth/2 + 1, base_depth/2 - 1]:
    for row in range(pattern_rows):
        row_height = 8 + row * 7
        offset_x = (row % 2) * diamond_size  # Stagger alternate rows
        
        for col in range(4):
            x_pos = -base_width/3 + col * diamond_size * 2 + offset_x
            
            if abs(x_pos) < base_width/2 - 5:  # Keep pattern within bounds
                # Create diamond shaped depression
                diamond = (cq.Workplane("XZ")
                    .workplane(offset=face_y)
                    .center(x_pos, row_height)
                    .polygon(4, diamond_size)
                    .extrude(-pattern_depth)
                    .rotate((0, 0, 0), (0, 0, 1), 45)
                )
                model = model.cut(diamond)

# Add vertical groove lines on side facets
for angle in [0, 90, 180, 270]:
    groove_x = (base_width/2 - 2) * math.cos(math.radians(angle))
    groove_y = (base_depth/2 - 2) * math.sin(math.radians(angle))
    
    if abs(angle - 90) < 45 or abs(angle - 270) < 45:  # Side facets only
        for i in range(3):
            groove = (cq.Workplane("XY")
                .workplane(offset=10 + i * 15)
                .center(groove_x * 0.9, groove_y * 0.9)
                .box(1.5, base_depth * 0.8, 1.0, centered=(True, True, False))
                .rotate((0, 0, 0), (0, 0, 1), angle)
            )
            model = model.cut(groove)

# Create decorative stopper cap
# Base of stopper that fits in neck
stopper = (cq.Workplane("XY")
    .workplane(offset=bottle_height + neck_height - 3)
    .circle(neck_radius - 0.5)
    .extrude(5)
)

# Decorative top part with Art Deco stepped pyramid design
stopper_decoration = (cq.Workplane("XY")
    .workplane(offset=bottle_height + neck_height + 2)
    .rect(stopper_base, stopper_base)
    .workplane(offset=3)
    .rect(stopper_base * 0.8, stopper_base * 0.8)
    .workplane(offset=3)
    .rect(stopper_base * 0.6, stopper_base * 0.6)
    .workplane(offset=3)
    .rect(stopper_base * 0.4, stopper_base * 0.4)
    .workplane(offset=2)
    .rect(stopper_base * 0.2, stopper_base * 0.2)
    .loft(combine=True)
)
stopper = stopper.union(stopper_decoration)

# Add radiating lines on stopper top
for i in range(8):
    angle = i * 45
    line = (cq.Workplane("XY")
        .workplane(offset=bottle_height + neck_height + cap_height)
        .box(stopper_top, 0.8, 0.5, centered=(True, True, False))
        .rotate((0, 0, 0), (0, 0, 1), angle)
    )
    stopper = stopper.cut(line)

model = model.union(stopper)

# Add small base indent for stability
base_indent = (cq.Workplane("XY")
    .rect(base_width - 10, base_depth - 10)
    .extrude(2)
)
model = model.cut(base_indent)

show(model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [17]:
#14
import cadquery as cq
from jupyter_cadquery import show
import math

# Heavy-Duty Pillow Block Bearing Housing - Industrial bearing mount with cooling fins
# Parameters
bearing_od = 52.0           # Outer diameter of bearing
bearing_width = 15.0        # Width of bearing
housing_width = 80.0        # Total housing width
housing_height = 65.0       # Total housing height
base_length = 110.0         # Base plate length
base_width = 60.0          # Base plate width
base_thickness = 12.0       # Base plate thickness

# Mounting parameters
bolt_hole_diameter = 8.5    # M8 bolt clearance holes
bolt_spacing = 85.0         # Distance between mounting bolts
corner_radius = 5.0         # Corner radius for base

# Cooling fin parameters
num_fins = 7                # Number of cooling fins
fin_thickness = 2.5         # Thickness of each fin
fin_height = 15.0           # Height of fins
fin_spacing = 4.0           # Space between fins

# Lubrication parameters
grease_nipple_dia = 6.0     # Grease nipple hole diameter
grease_channel_dia = 3.0    # Internal grease channel diameter

# Create base mounting plate with rounded corners
model = (cq.Workplane("XY")
    .box(base_length, base_width, base_thickness)
    .edges("|Z").fillet(corner_radius)
)

# Add reinforcement ribs under base
for x_offset in [-25, 25]:
    rib = (cq.Workplane("XZ")
        .workplane(offset=0)
        .center(x_offset, base_thickness/2)
        .polyline([(0, 0), (15, 0), (0, 8)])
        .close()
        .extrude(4)
    )
    model = model.union(rib)

# Create main bearing housing block
housing_block = (cq.Workplane("XY")
    .workplane(offset=base_thickness)
    .circle(housing_height/2)
    .extrude(housing_width)
)

# Cut bearing bore through housing
bearing_bore = (cq.Workplane("YZ")
    .workplane(offset=0)
    .center(base_thickness + housing_height/2, housing_width/2)
    .circle(bearing_od/2)
    .extrude(base_length, both=True)
)
model = model.union(housing_block).cut(bearing_bore)

# Add split cap design - cut top half for bearing installation
split_height = base_thickness + housing_height/2 + 5
split_cut = (cq.Workplane("XY")
    .workplane(offset=split_height)
    .box(base_length, housing_width + 10, housing_height)
)
model = model.cut(split_cut)

# Create bearing cap (top half)
cap = (cq.Workplane("XY")
    .workplane(offset=split_height)
    .center(0, 0)
    .box(housing_height, housing_width, housing_height/2 - 5)
)

# Cut bearing bore in cap
cap_bore = (cq.Workplane("YZ")
    .workplane(offset=0)
    .center(split_height + (housing_height/2 - 5)/2, housing_width/2)
    .circle(bearing_od/2)
    .extrude(housing_height, both=True)
)
cap = cap.cut(cap_bore)

# Add cap bolt bosses
for x in [-20, 20]:
    for y in [-housing_width/2 + 10, housing_width/2 - 10]:
        boss = (cq.Workplane("XY")
            .workplane(offset=split_height)
            .center(x, y)
            .circle(7)
            .extrude(housing_height/2 - 5)
        )
        cap = cap.union(boss)
        
        # Add bolt holes
        bolt_hole = (cq.Workplane("XY")
            .workplane(offset=split_height - 1)
            .center(x, y)
            .circle(3)
            .extrude(housing_height)
        )
        cap = cap.cut(bolt_hole)
        model = model.cut(bolt_hole)  # Matching holes in base housing

# Add cooling fins to housing sides
fin_start_y = -num_fins * (fin_thickness + fin_spacing) / 2
for i in range(num_fins):
    y_pos = fin_start_y + i * (fin_thickness + fin_spacing)
    
    # Left side fins
    fin_left = (cq.Workplane("XZ")
        .workplane(offset=y_pos)
        .center(-housing_height/2 - fin_height/2, base_thickness + housing_height/2)
        .box(fin_height, housing_height * 0.7, fin_thickness)
    )
    
    # Right side fins  
    fin_right = (cq.Workplane("XZ")
        .workplane(offset=y_pos)
        .center(housing_height/2 + fin_height/2, base_thickness + housing_height/2)
        .box(fin_height, housing_height * 0.7, fin_thickness)
    )
    
    model = model.union(fin_left).union(fin_right)
    cap = cap.union(fin_left.translate((0, 0, split_height - base_thickness - housing_height/2)))
    cap = cap.union(fin_right.translate((0, 0, split_height - base_thickness - housing_height/2)))

# Add mounting bolt holes in base
for x in [-bolt_spacing/2, bolt_spacing/2]:
    for y in [-20, 20]:
        # Counterbored hole
        hole = (cq.Workplane("XY")
            .center(x, y)
            .circle(bolt_hole_diameter/2)
            .extrude(base_thickness + 1)
        )
        
        # Counterbore for bolt head
        cbore = (cq.Workplane("XY")
            .workplane(offset=base_thickness - 4)
            .center(x, y)
            .circle(bolt_hole_diameter)
            .extrude(5)
        )
        model = model.cut(hole).cut(cbore)

# Add grease nipple fitting on top
grease_fitting = (cq.Workplane("XY")
    .workplane(offset=split_height + housing_height/2 - 8)
    .circle(grease_nipple_dia/2)
    .extrude(8)
)
cap = cap.cut(grease_fitting)

# Add grease channel to bearing
grease_channel = (cq.Workplane("XY")
    .workplane(offset=split_height)
    .circle(grease_channel_dia/2)
    .extrude(housing_height/2)
)
cap = cap.cut(grease_channel)

# Add part number and size stamping (as recessed text area)
text_recess = (cq.Workplane("XZ")
    .workplane(offset=housing_width - 2)
    .center(0, base_thickness + 15)
    .box(30, 8, 1)
)
model = model.cut(text_recess)

# Combine cap with main housing
model = model.union(cap)

# Add oil drain plug location
drain_plug = (cq.Workplane("YZ")
    .workplane(offset=0)
    .center(base_thickness + 5, housing_width/2)
    .circle(4)
    .extrude(10)
)
model = model.cut(drain_plug)

show(model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [18]:
#15
import cadquery as cq
from jupyter_cadquery import show
import math

# Industrial Jaw Coupling - Flexible shaft coupler with spider element
# Parameters
coupling_od = 65.0          # Outer diameter of coupling
coupling_length = 75.0      # Total length of coupling
shaft_bore = 20.0          # Shaft bore diameter
keyway_width = 6.0         # Keyway width
keyway_depth = 3.5         # Keyway depth into shaft

# Jaw parameters
num_jaws = 3               # Number of coupling jaws
jaw_angle = 120.0          # Angle between jaws
jaw_thickness = 18.0       # Thickness of each jaw
jaw_clearance = 2.0        # Clearance for spider

# Spider insert parameters
spider_thickness = 25.0     # Thickness of flexible element
spider_od = 52.0           # Outer diameter of spider
spider_hardness = 4.0      # Width of spider arms

# Hub parameters
hub_length = 30.0          # Length of each hub
flange_thickness = 8.0     # Coupling flange thickness
hub_taper_length = 10.0    # Taper transition length

# Set screw parameters
setscrew_dia = 6.0         # M6 set screw
setscrew_offset = 12.0     # Distance from end

# Create main coupling hub body
model = (cq.Workplane("XY")
    .circle(coupling_od/2)
    .extrude(hub_length)
)

# Add tapered transition section
taper_section = (cq.Workplane("XY")
    .workplane(offset=hub_length)
    .circle(coupling_od/2)
    .workplane(offset=hub_taper_length)
    .circle(coupling_od/2 - 3)
    .loft()
)
model = model.union(taper_section)

# Create jaw section base
jaw_section = (cq.Workplane("XY")
    .workplane(offset=hub_length + hub_taper_length)
    .circle(coupling_od/2 - 3)
    .extrude(coupling_length - hub_length - hub_taper_length)
)
model = model.union(jaw_section)

# Cut shaft bore through entire length
shaft_bore_cut = (cq.Workplane("XY")
    .circle(shaft_bore/2)
    .extrude(coupling_length + 1)
)
model = model.cut(shaft_bore_cut)

# Add keyway slot
keyway = (cq.Workplane("XY")
    .center(0, shaft_bore/2 - keyway_depth/2)
    .box(keyway_width, keyway_depth + 1, hub_length)
)
model = model.cut(keyway)

# Create driving jaws with curved profile
for i in range(num_jaws):
    angle = i * jaw_angle
    
    # Calculate jaw geometry
    jaw_center_radius = (coupling_od/2 + spider_od/2) / 2
    jaw_x = jaw_center_radius * math.cos(math.radians(angle))
    jaw_y = jaw_center_radius * math.sin(math.radians(angle))
    
    # Create jaw protrusion
    jaw = (cq.Workplane("XY")
        .workplane(offset=coupling_length - jaw_thickness - 5)
        .center(jaw_x, jaw_y)
        .box(coupling_od/2 - spider_od/2 + 5, jaw_thickness, jaw_thickness)
        .rotate((0, 0, 0), (0, 0, 1), angle)
    )
    model = model.union(jaw)
    
    # Add curved contact surface
    contact_curve = (cq.Workplane("XY")
        .workplane(offset=coupling_length - jaw_thickness - 5)
        .center(spider_od/2 * math.cos(math.radians(angle + jaw_angle/2)), 
                spider_od/2 * math.sin(math.radians(angle + jaw_angle/2)))
        .circle(jaw_thickness/2)
        .extrude(jaw_thickness)
    )
    model = model.cut(contact_curve)

# Create spider cavity
spider_cavity = (cq.Workplane("XY")
    .workplane(offset=coupling_length - spider_thickness - 8)
    .circle(spider_od/2 + jaw_clearance)
    .extrude(spider_thickness + jaw_clearance)
)
model = model.cut(spider_cavity)

# Add set screw holes
for angle in [0, 120]:
    setscrew_x = (coupling_od/2) * math.cos(math.radians(angle))
    setscrew_y = (coupling_od/2) * math.sin(math.radians(angle))
    
    # Threaded hole for set screw
    setscrew_hole = (cq.Workplane("XZ")
        .workplane(offset=setscrew_y)
        .center(setscrew_x, setscrew_offset)
        .circle(setscrew_dia/2)
        .extrude(coupling_od/2)
        .rotate((0, 0, setscrew_offset), (0, 1, 0), angle)
    )
    model = model.cut(setscrew_hole)

# Add oil grooves for lubrication
for i in range(4):
    groove_angle = i * 90
    groove = (cq.Workplane("XY")
        .workplane(offset=hub_length/2)
        .center((shaft_bore/2 + 2) * math.cos(math.radians(groove_angle)),
                (shaft_bore/2 + 2) * math.sin(math.radians(groove_angle)))
        .box(2, 8, hub_length * 0.8)
        .rotate((0, 0, 0), (0, 0, 1), groove_angle)
    )
    model = model.cut(groove)

# Add balance holes for high-speed operation
for i in range(6):
    angle = i * 60
    balance_r = coupling_od/2 - 8
    hole_x = balance_r * math.cos(math.radians(angle))
    hole_y = balance_r * math.sin(math.radians(angle))
    
    balance_hole = (cq.Workplane("XY")
        .workplane(offset=5)
        .center(hole_x, hole_y)
        .circle(3)
        .extrude(8)
    )
    model = model.cut(balance_hole)

# Add identification groove (for size/type marking)
id_groove = (cq.Workplane("XY")
    .workplane(offset=hub_length - 5)
    .circle(coupling_od/2)
    .circle(coupling_od/2 - 1)
    .extrude(2)
)
model = model.cut(id_groove)

# Add chamfers to shaft entrance
model = model.edges("<Z").edges(">X").chamfer(2)

show(model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [19]:
#16
import cadquery as cq
from jupyter_cadquery import show
import math

# Industrial Gusseted Mounting Bracket — L-bracket with ribs, slots, and countersunk holes

# -----------------------
# Parameters (edit freely)
# -----------------------
base_length = 120.0       # X size of the base plate
base_width  = 60.0        # Y size of the base plate
wall_height = 80.0        # Z height of the vertical flange
thickness   = 6.0         # Plate thickness

outer_fillet = 3.0        # Fillet for external vertical edges
edge_chamfer = 1.2        # Chamfer on flange top rim
safe_fillet  = min(outer_fillet, 0.49*thickness)  # robust radius to avoid OCC failures

mount_hole_d    = 6.0     # Through hole diameter on base
csk_diameter    = 12.0    # Countersink diameter
csk_angle       = 82.0    # Countersink angle (deg)
mount_cols      = 3       # Count along X
mount_pitch_x   = 40.0    # Spacing along X
mount_offset_x  = -40.0   # First column offset from center

slot_length   = 18.0      # Slots in the vertical flange (Z direction)
slot_width    = 6.0
slot_rows     = 2
slot_z_gap    = 24.0      # Spacing between slot rows
slot_x_inset  = 25.0      # Inset from left/right along X

lighten_count      = 6    # Lightening holes in an arc on the base
lighten_d          = 10.0
arc_center_offset  = 18.0 # Arc center offset from the back-right corner
arc_radius         = 22.0 # Arc radius for lightening holes
arc_span_deg       = 90.0 # Total span of the arc

rib_length    = 55.0      # Rib triangle base along X
rib_height    = 45.0      # Rib triangle height along Z
rib_thickness = 6.0       # Rib thickness (extrude along Y)
rib_y_clear   = 8.0       # Clearance from back corner towards -Y

# ------------------------------------
# Main geometry: build parts, treat edges, then assemble
# ------------------------------------
# Base plate on XY (build edge treatments BEFORE union to avoid seam issues)
base = (
    cq.Workplane("XY")
    .box(base_length, base_width, thickness, centered=(True, True, False))
    # Soften only vertical perimeter of the base (robust selection)
    .edges("|Z").fillet(safe_fillet)
)

# Vertical flange along +Y edge, rising 'wall_height'
flange = (
    cq.Workplane("XY")
    .box(base_length, thickness, wall_height + thickness, centered=(True, True, False))
    .translate((0, base_width/2 - thickness/2, 0))
)
# Fillet ONLY the two external vertical edges of the outer face to avoid self-intersections
flange = flange.faces(">Y").edges("|Z").fillet(safe_fillet)
# Chamfer the flange's top perimeter for de-burring
flange = flange.faces(">Z").edges().chamfer(edge_chamfer)

# Union to form the L-bracket
bracket = base.union(flange)

# ------------------------------------
# Mounting pattern with countersinks (parametric array)
# ------------------------------------
x_positions = [mount_offset_x + i * mount_pitch_x for i in range(mount_cols)]
bracket = (
    bracket
    .faces(">Z").workplane(centerOption="CenterOfMass")   # top of base
    .pushPoints([(x, 0) for x in x_positions])
    .cskHole(mount_hole_d, csk_diameter, csk_angle)       # countersunk through-holes
)

# ------------------------------------
# Slots on vertical flange (practical adjustability)
# ------------------------------------
slot_pts = []
z0 = wall_height*0.35
for r in range(slot_rows):
    z = z0 + r * slot_z_gap
    slot_pts += [(-slot_x_inset, z), (slot_x_inset, z)]  # symmetric columns about X=0

bracket = (
    bracket
    .faces(">Y").workplane(centerOption="CenterOfMass")   # outside of flange; coords are (X,Z)
    .pushPoints(slot_pts)
    .slot2D(slot_length, slot_width)                      # create slot sketches
    .cutThruAll()                                         # cut fully through flange
)

# ------------------------------------
# Lightening holes in an arc (uses trig for circular positioning)
# ------------------------------------
arc_cx =  base_length/2 - arc_center_offset
arc_cy =  base_width/2  - arc_center_offset
step = arc_span_deg / (lighten_count - 1 if lighten_count > 1 else 1)
arc_points = []
for i in range(lighten_count):
    a = math.radians(i * step)
    px = arc_cx - arc_radius * math.cos(a)
    py = arc_cy - arc_radius * math.sin(a)
    arc_points.append((px, py))

bracket = (
    bracket
    .faces(">Z").workplane(origin=(0, 0, thickness))      # top face level
    .pushPoints(arc_points)
    .hole(lighten_d)
)

# ------------------------------------
# Triangular ribs (iterative, mirrored) to stiffen bracket corner
# ------------------------------------
rib = (
    cq.Workplane("XZ")
    .polyline([(0, 0), (rib_length, 0), (0, rib_height)]).close()
    .extrude(-rib_thickness)                              # into -Y
    .translate((-base_length/2 + rib_length + 6.0,
                base_width/2 - rib_y_clear,
                thickness))
)
# Two ribs per side, then mirror across X=0 for symmetry; fillet rib edges for stress relief
ribs = (
    rib.union(rib.translate((20.0, 0, 0)))
       .mirror("YZ", union=True)
       .edges("|Z or |X").fillet(2.0)
)
bracket = bracket.union(ribs)

# ------------------------------------
# Final display
# ------------------------------------
model = bracket
show(model)


+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [20]:
#17 
import cadquery as cq
from jupyter_cadquery import show
import math

# V-Belt Pulley with Spokes — realistic industrial pulley with V-groove, hub, spokes, and bolt circle

# -----------------------
# Parameters
# -----------------------
rim_radius        = 50.0     # Outer radius of the rim
rim_width         = 20.0     # Axial width of the rim
rim_edge_fillet   = 1.2      # Edge fillet on rim edges

v_angle_deg       = 40.0     # Half-angle of the V-groove (total angle = 2*half)
v_depth           = 6.0      # Depth of the V-groove into the rim
v_mid_offset      = 0.0      # Shift groove radially (+ outwards, - inwards)

hub_radius        = 14.0     # Hub outer radius
hub_length        = rim_width + 8.0  # Hub sticks out beyond rim a bit
bore_diameter     = 10.0     # Shaft bore through the hub

spoke_count       = 6        # Number of spokes
spoke_thickness   = 5.0      # Circumferential thickness of each spoke
spoke_height      = rim_width # Spokes span the full rim width
spoke_root_rad    = hub_radius + 3.0
spoke_tip_rad     = rim_radius - 6.0
spoke_fillet      = 1.0

bolt_circle_rad   = hub_radius + 10.0
bolt_diameter     = 5.0
bolt_count        = 3

# -----------------------
# Main geometry
# -----------------------
# Rim as a cylinder centered on Z
pulley = cq.Workplane("XY").cylinder(rim_width, rim_radius, centered=True)

# Smooth rim edges for realism
pulley = pulley.edges(">Z or <Z").fillet(rim_edge_fillet)

# V-groove cutter: revolve a triangular profile around Z to create a 360° groove
half_ang = math.radians(v_angle_deg)
apex_x = rim_radius - v_depth + v_mid_offset
delta  = v_depth * math.tan(half_ang)
v_prof = (
    cq.Workplane("XZ")
    .moveTo(apex_x, 0)
    .lineTo(apex_x + delta,  v_depth)    # one flank
    .lineTo(apex_x - delta,  v_depth)    # other flank
    .close()
    .revolve(360, (0, 0, 0), (0, 0, 1))  # spin around global Z
)
pulley = pulley.cut(v_prof)              # boolean cut for the V groove

# Hub (union) and bore (cut)
hub = cq.Workplane("XY").cylinder(hub_length, hub_radius, centered=True)
pulley = pulley.union(hub)
pulley = pulley.hole(bore_diameter)      # through all along Z

# Spokes: rectangular ribs from hub to rim, patterned radially
spoke_len = (spoke_tip_rad - spoke_root_rad)
one_spoke = (
    cq.Workplane("XY")
    .center((spoke_root_rad + spoke_tip_rad)/2.0, 0)   # center between root & tip on +X
    .rect(spoke_len, spoke_thickness)
    .extrude(spoke_height, both=True)                  # full width symmetrically
    .edges("|Z or |X").fillet(spoke_fillet)
)

# Radial array using a loop (iterative + trig-based positioning)
for i in range(spoke_count):
    ang = i * 360.0 / spoke_count
    pulley = pulley.union(one_spoke.rotate((0, 0, 0), (0, 0, 1), ang))

# Bolt circle on both hub faces (practical feature)
bolt_pts = [
    (bolt_circle_rad * math.cos(2*math.pi*i/bolt_count),
     bolt_circle_rad * math.sin(2*math.pi*i/bolt_count))
    for i in range(bolt_count)
]
pulley = (
    pulley.faces(">Z").workplane(centerOption="CenterOfMass")
    .pushPoints(bolt_pts).hole(bolt_diameter, depth=hub_length/2 + 0.02)
    .faces("<Z").workplane(centerOption="CenterOfMass")
    .pushPoints(bolt_pts).hole(bolt_diameter, depth=hub_length/2 + 0.02)
)

# Final small fillet at the hub/face transition for stress relief (safe selection)
pulley = pulley.faces(">Z or <Z").edges("%Line").fillet(0.5)

# -----------------------
# Final display
# -----------------------
model = pulley
show(model)


+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [21]:
#18 
import cadquery as cq
from jupyter_cadquery import show
import math  # not strictly needed, but available

# Door Hinge Assembly — two leaves with interleaved knuckles, pin bore, and countersunk holes

# -----------------------
# Parameters (edit freely)
# -----------------------
leaf_length        = 90.0      # Overall hinge length along X
leaf_width         = 35.0      # Leaf width along Y (each leaf ~ half this, minus barrel)
leaf_thickness     = 3.0       # Leaf plate thickness (Z)
barrel_outer_d     = 10.0      # Knuckle OD
barrel_inner_d     = 5.0       # Pin/bore diameter through knuckles
knuckle_count      = 5         # Total knuckles (odd preferred: center belongs to left leaf)
knuckle_gap        = 2.0       # Axial gap between knuckles along X
leaf_clear_to_bar  = 0.6       # Clearance between barrel and leaf edge (visual gap)
hole_d             = 4.0       # Through/countersunk hole diameter on leaves
csk_d              = 8.0       # Countersink diameter
csk_angle          = 82.0      # Countersink angle
edge_chamfer       = 0.6       # Chamfer around leaf perimeter
barrel_edge_fillet = 0.7       # Fillet around barrel segment ends

# -----------------------
# Derived helpers
# -----------------------
barrel_r = barrel_outer_d/2
pin_r    = barrel_inner_d/2
usable_len = leaf_length - (knuckle_count-1)*knuckle_gap
knuckle_len = usable_len/knuckle_count
half_w  = leaf_width/2
bar_y   = 0.0                        # Barrel axis along X at Y=0
leaf_y_offset = barrel_r + leaf_clear_to_bar + (half_w - barrel_r - leaf_clear_to_bar)/2

# -----------------------
# Build two leaf plates (left/right) with a small clearance from the barrel line
# -----------------------
left_leaf = (
    cq.Workplane("XY")
    .center(0, - (barrel_r + leaf_clear_to_bar + (half_w - barrel_r - leaf_clear_to_bar)/2))
    .rect(leaf_length, half_w - barrel_r - leaf_clear_to_bar)
    .extrude(leaf_thickness)
    .edges("|Z").chamfer(edge_chamfer)
)
right_leaf = (
    cq.Workplane("XY")
    .center(0,  (barrel_r + leaf_clear_to_bar + (half_w - barrel_r - leaf_clear_to_bar)/2))
    .rect(leaf_length, half_w - barrel_r - leaf_clear_to_bar)
    .extrude(leaf_thickness)
    .edges("|Z").chamfer(edge_chamfer)
)

# -----------------------
# Interleaved knuckles along X (cylinders on YZ plane extruded along X)
# Alternate union to left/right leaf using index parity
# -----------------------
x0 = -leaf_length/2 + knuckle_len/2
for i in range(knuckle_count):
    cx = x0 + i*(knuckle_len + knuckle_gap)
    seg = (
        cq.Workplane("YZ")
        .center(bar_y, leaf_thickness/2)                 # place at Y=0, on plate mid-Z
        .circle(barrel_r)
        .extrude(knuckle_len, both=False)
        .translate((cx - knuckle_len/2, 0, 0))           # align segment about its center at cx
        .faces(">X or <X").edges("%Circle").fillet(barrel_edge_fillet)
    )
    if i % 2 == 0:
        left_leaf = left_leaf.union(seg)
    else:
        right_leaf = right_leaf.union(seg)

# Bore the pin hole through all knuckles (boolean cut with long cylinder)
pin_bore = cq.Workplane("YZ").center(0, leaf_thickness/2).circle(pin_r).extrude(leaf_length, both=True)
left_leaf = left_leaf.cut(pin_bore)
right_leaf = right_leaf.cut(pin_bore)

# -----------------------
# Practical mounting: countersunk holes on each leaf in a 2x2 array
# -----------------------
x_pitch = leaf_length*0.33
y_inset = (half_w - barrel_r - 2*leaf_clear_to_bar)*0.4
leaf_hole_pts = [(-x_pitch/2, -y_inset), (x_pitch/2, -y_inset),
                 (-x_pitch/2,  y_inset), (x_pitch/2,  y_inset)]

left_leaf = (
    left_leaf.faces(">Z").workplane(centerOption="CenterOfMass")
    .pushPoints(leaf_hole_pts).cskHole(hole_d, csk_d, csk_angle)
)
right_leaf = (
    right_leaf.faces(">Z").workplane(centerOption="CenterOfMass")
    .pushPoints([(x, -y) for (x, y) in leaf_hole_pts])   # mirror pattern across Y=0
    .cskHole(hole_d, csk_d, csk_angle)
)

# -----------------------
# Final touches: slight edge fillet at barrel/leaf junction for realism
# -----------------------
left_leaf  = left_leaf.edges(">Y or <Y").fillet(0.3)
right_leaf = right_leaf.edges(">Y or <Y").fillet(0.3)

# -----------------------
# Assemble and display
# -----------------------
model = left_leaf.union(right_leaf)
show(model)


+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [22]:
#19
import cadquery as cq
from jupyter_cadquery import show
import math

# Tool Organizer Tray — ribbed storage tray with grid compartments, finger notches, feet, and magnet pockets

# -----------------------
# Parameters (edit freely)
# -----------------------
outer_length      = 160.0   # X size of tray
outer_width       = 100.0   # Y size of tray
outer_height      = 28.0    # Z height of tray
wall_thickness    = 3.0     # Side wall thickness
base_thickness    = 3.0     # Bottom wall thickness
corner_radius     = 6.0     # Outside vertical edge fillet (safe < min/2)

cols              = 3       # Number of compartments across X
rows              = 2       # Number of compartments across Y
rib_thickness     = 2.5     # Internal rib thickness between cells

finger_notch_d    = 18.0    # Semicircular finger notch diameter on front wall
label_recess_w    = 40.0    # Label slot width on front
label_recess_h    = 8.0     # Label slot height
label_recess_d    = 1.0     # Label recess depth

foot_diam         = 10.0    # Round feet on underside
foot_height       = 1.6
foot_inset        = 8.0

mag_pocket_d      = 8.0     # Cylindrical magnet pockets under base
mag_pocket_h      = 2.2
mag_ring_radius   = 30.0    # Radial placement circle for 4 pockets

rim_chamfer       = 0.6     # De-burr the top rim (small chamfer)

# -----------------------
# Outer shell and cavity (Boolean cut, robust)
# -----------------------
tray = (
    cq.Workplane("XY")
    .box(outer_length, outer_width, outer_height, centered=(True, True, False))
    .edges("|Z").fillet(min(corner_radius, min(outer_length, outer_width)/4))
)

inner_len = outer_length - 2*wall_thickness
inner_wid = outer_width  - 2*wall_thickness
inner_h   = outer_height - base_thickness

cavity = (
    cq.Workplane("XY")
    .box(inner_len, inner_wid, inner_h, centered=(True, True, False))
    .translate((0, 0, base_thickness))
)
tray = tray.cut(cavity)

# -----------------------
# Grid ribs (iterative construction using loops) — UNION (add material)
# -----------------------
cell_pitch_x = inner_len/cols
cell_pitch_y = inner_wid/rows

# Vertical ribs across Y (between columns)
for c in range(1, cols):
    x = -inner_len/2 + c*cell_pitch_x
    rib = (
        cq.Workplane("XY")
        .box(rib_thickness, inner_wid, inner_h, centered=(True, True, False))
        .translate((x, 0, base_thickness))
    )
    tray = tray.union(rib)

# Horizontal ribs across X (between rows)
for r in range(1, rows):
    y = -inner_wid/2 + r*cell_pitch_y
    rib = (
        cq.Workplane("XY")
        .box(inner_len, rib_thickness, inner_h, centered=(True, True, False))
        .translate((0, y, base_thickness))
    )
    tray = tray.union(rib)

# -----------------------
# Finger notches & label recess on the front wall (practical features)
# -----------------------
front_y = -outer_width/2 + wall_thickness/2
for c in range(cols):
    cx = -inner_len/2 + (c+0.5)*cell_pitch_x
    notch = (
        cq.Workplane("YZ")
        .center(front_y, base_thickness + inner_h*0.5)
        .circle(finger_notch_d/2)
        .extrude(outer_length, both=True)
        .translate((cx, 0, 0))
    )
    tray = tray.cut(notch)

label = (
    cq.Workplane("XY")
    .center(0, front_y + wall_thickness/2)
    .rect(label_recess_w, label_recess_h)
    .extrude(label_recess_d)
    .translate((0, 0, outer_height*0.45))
)
tray = tray.cut(label)

# -----------------------
# Round feet (unions) at the underside corners
# -----------------------
foot_r = foot_diam/2
for sx in (-1, 1):
    for sy in (-1, 1):
        fx = sx*(outer_length/2 - foot_inset)
        fy = sy*(outer_width/2  - foot_inset)
        foot = cq.Workplane("XY").center(fx, fy).circle(foot_r).extrude(foot_height)
        tray = tray.union(foot)

# -----------------------
# Magnet pockets (trig placement around a circle, cuts)
# -----------------------
for i in range(4):
    ang = 2*math.pi*i/4
    px = mag_ring_radius*math.cos(ang)
    py = mag_ring_radius*math.sin(ang)
    pocket = cq.Workplane("XY").center(px, py).circle(mag_pocket_d/2).extrude(mag_pocket_h)
    tray = tray.cut(pocket)

# -----------------------
# Top rim chamfer (edge treatment; small and safe)
# -----------------------
tray = tray.faces(">Z").edges("|X or |Y").chamfer(rim_chamfer)

# -----------------------
# Final display
# -----------------------
model = tray
show(model)


+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [23]:
#20
import cadquery as cq
from jupyter_cadquery import show
import math

# Decorative Bowl with Cutouts — lofted shell, rim bead, polar cutouts, and vertical flutes
# Robust: no fragile edge selections on the rim

# -----------------------
# Parameters
# -----------------------
height            = 70.0      # Overall height
base_radius       = 18.0      # Outer radius at the base
belly_radius      = 55.0      # Max outer radius
rim_radius        = 50.0      # Outer radius at the rim
wall_thickness    = 3.0       # Shell thickness

rim_bead_r        = 1.6       # Torus bead at the rim

foot_height       = 2.0       # Bottom ring foot height
foot_overhang     = 3.0       # Foot outer radius grows beyond base
foot_fillet       = 0.8

band_z1           = 30.0      # First cutout band height
band_z2           = 48.0      # Second cutout band height
cut_count         = 18        # Cutouts around circumference
cut_radius        = 3.0       # Each round cut radius

flute_count       = 8         # Vertical flutes around body
flute_width       = 10.0      # Slot width (chord, at belly)
flute_depth       = 2.5       # Radial cut depth

# -----------------------
# Helper: radius at a given Z (piecewise linear)
# -----------------------
stations = [
    (0.0,            base_radius),
    (height*0.35,    belly_radius),
    (height,         rim_radius),
]
def radius_at(z):
    r = stations[-1][1]
    for (z0, r0), (z1, r1) in zip(stations[:-1], stations[1:]):
        if z0 <= z <= z1:
            t = (z - z0)/(z1 - z0)
            r = (1-t)*r0 + t*r1
            break
    return r

# -----------------------
# Main shell by loft + subtraction (robust vs shell op)
# -----------------------
wp = cq.Workplane("XY").circle(stations[0][1])
zprev = stations[0][0]
for z, r in stations[1:]:
    wp = wp.workplane(offset=z - zprev).circle(r); zprev = z
outer = wp.loft(combine=True)

wp_i = cq.Workplane("XY").circle(max(stations[0][1]-wall_thickness, 1.0))
zprev = stations[0][0]
for z, r in stations[1:]:
    wp_i = wp_i.workplane(offset=z - zprev).circle(max(r-wall_thickness, 1.0)); zprev = z
inner = wp_i.loft(combine=True)

bowl = outer.cut(inner)

# -----------------------
# Rim bead (torus by revolve) — smooth lip without extra fillet
# -----------------------
bead = (cq.Workplane("XZ")
        .moveTo(rim_radius, height)
        .circle(rim_bead_r)
        .revolve(360, (0,0,0), (0,0,1)))
bowl = bowl.union(bead)

# -----------------------
# Foot ring at base (union) with safe fillet
# -----------------------
foot_outer = base_radius + foot_overhang
foot_inner = max(foot_outer - (foot_overhang*1.6), 2.0)
foot = (cq.Workplane("XY")
        .circle(foot_outer).circle(foot_inner)
        .extrude(foot_height)
        .faces(">Z or <Z").edges("%Circle").fillet(min(foot_fillet, foot_height/2 - 0.01)))
bowl = bowl.union(foot)

# -----------------------
# Polar round cutouts in two bands (robust cylindrical cutters)
# -----------------------
for band_z in (band_z1, band_z2):
    band_r = radius_at(band_z) - wall_thickness/2
    cutter_len = band_r*2 + 6.0
    unit_cutter = (cq.Workplane("YZ").center(0, band_z)
                   .circle(cut_radius).extrude(cutter_len, both=True))
    for i in range(cut_count):
        ang = 360.0*i/cut_count
        bowl = bowl.cut(unit_cutter.rotate((0,0,0),(0,0,1), ang).translate((band_r,0,0)))

# -----------------------
# Vertical flutes (slots) around the belly — array by rotation
# -----------------------
belly_z = stations[1][0]
belly_r = radius_at(belly_z) - wall_thickness/2
flute = (cq.Workplane("XY")
         .center(belly_r, 0)
         .rect(flute_depth*2, flute_width)
         .extrude(height))
for i in range(flute_count):
    ang = 360.0*i/flute_count
    bowl = bowl.cut(flute.rotate((0,0,0),(0,0,1), ang))

# -----------------------
# Final display
# -----------------------
model = bowl
show(model)


+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [24]:
#21
import cadquery as cq
from jupyter_cadquery import show
import math

# Spur Gear - Used in power transmission systems
# Simplified reliable version

# Gear parameters
module = 3  # Gear module (tooth size)
num_teeth = 20  # Reduced for better performance
pressure_angle = 20  # degrees
face_width = 20
hub_diameter = 25
hub_length = 30
bore_diameter = 12
keyway_width = 4
keyway_depth = 2

# Calculate gear dimensions
pitch_diameter = module * num_teeth
outer_diameter = pitch_diameter + 2 * module
root_diameter = pitch_diameter - 2.5 * module

# Create gear blank (main body)
gear = cq.Workplane("XY").circle(outer_diameter/2).extrude(face_width)

# Create hub
hub = cq.Workplane("XY").circle(hub_diameter/2).extrude(hub_length)
hub = hub.translate((0, 0, face_width))
gear = gear.union(hub)

# Create bore hole
bore = cq.Workplane("XY").circle(bore_diameter/2).extrude(hub_length + face_width + 1)
bore = bore.translate((0, 0, -0.5))
gear = gear.cut(bore)

# Create keyway
keyway = cq.Workplane("XY").box(keyway_width, bore_diameter + 2, hub_length + face_width + 1)
keyway = keyway.translate((0, 0, (hub_length + face_width)/2 - 0.5))
gear = gear.cut(keyway)

# Create gear teeth using simple wedge cutters
tooth_angle = 360 / num_teeth
tooth_space_angle = tooth_angle * 0.45  # Width of space between teeth

for i in range(num_teeth):
    angle = tooth_angle * i
    
    # Create wedge-shaped tooth space cutter
    cutter_inner = root_diameter / 2
    cutter_outer = outer_diameter / 2 + 2
    half_angle = tooth_space_angle / 2
    
    # Calculate wedge points
    p1 = (cutter_inner * math.cos(math.radians(-half_angle)), 
          cutter_inner * math.sin(math.radians(-half_angle)))
    p2 = (cutter_outer * math.cos(math.radians(-half_angle)), 
          cutter_outer * math.sin(math.radians(-half_angle)))
    p3 = (cutter_outer * math.cos(math.radians(half_angle)), 
          cutter_outer * math.sin(math.radians(half_angle)))
    p4 = (cutter_inner * math.cos(math.radians(half_angle)), 
          cutter_inner * math.sin(math.radians(half_angle)))
    
    # Create and extrude wedge
    cutter = cq.Workplane("XY").moveTo(p1[0], p1[1])
    cutter = cutter.lineTo(p2[0], p2[1])
    cutter = cutter.lineTo(p3[0], p3[1])
    cutter = cutter.lineTo(p4[0], p4[1])
    cutter = cutter.close().extrude(face_width + 2)
    cutter = cutter.translate((0, 0, -1))
    cutter = cutter.rotate((0, 0, 0), (0, 0, 1), angle)
    
    gear = gear.cut(cutter)

# Add lightening holes in web
lightening_hole_diameter = 6
lightening_circle_diameter = (hub_diameter + root_diameter) / 2
num_lightening_holes = 6

for i in range(num_lightening_holes):
    angle = (360 / num_lightening_holes) * i + 15
    x = (lightening_circle_diameter/2) * math.cos(math.radians(angle))
    y = (lightening_circle_diameter/2) * math.sin(math.radians(angle))
    
    hole = cq.Workplane("XY").circle(lightening_hole_diameter/2).extrude(face_width + 1)
    hole = hole.translate((x, y, -0.5))
    gear = gear.cut(hole)

# Add set screw holes in hub
set_screw_diameter = 3
set_screw_depth = 8
num_set_screws = 3

for i in range(num_set_screws):
    angle = (360 / num_set_screws) * i
    x = (hub_diameter/2 - 1) * math.cos(math.radians(angle))
    y = (hub_diameter/2 - 1) * math.sin(math.radians(angle))
    
    # Radial hole
    screw_hole = cq.Workplane("XY").circle(set_screw_diameter/2).extrude(set_screw_depth)
    screw_hole = screw_hole.rotate((0, 0, 0), (0, 1, 0), 90)
    screw_hole = screw_hole.rotate((0, 0, 0), (0, 0, 1), angle)
    screw_hole = screw_hole.translate((x * 0.5, y * 0.5, face_width + hub_length/2))
    gear = gear.cut(screw_hole)

show(gear)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [25]:
#22
import cadquery as cq
from jupyter_cadquery import show
import math

# Deep Groove Ball Bearing - Common rotating element bearing
# Simplified version that works reliably

# Bearing parameters
bore_diameter = 20  # Inner diameter
outer_diameter = 47  # Outer diameter
width = 14  # Bearing width
ball_diameter = 7.5  # Ball size
num_balls = 9  # Number of balls

# Race parameters
inner_race_wall = 3.5
outer_race_wall = 3.5
groove_depth = 2

# Calculate pitch diameter (where balls sit)
pitch_diameter = (bore_diameter + outer_diameter) / 2

# Create outer race
outer_race = cq.Workplane("XY").circle(outer_diameter/2).extrude(width)

# Create outer groove (simplified as cylinder cut)
outer_groove_diameter = outer_diameter - outer_race_wall * 2 - ball_diameter/2
outer_groove = cq.Workplane("XY").circle(outer_groove_diameter/2 + groove_depth).extrude(ball_diameter * 0.9)
outer_groove = outer_groove.translate((0, 0, width/2 - ball_diameter * 0.45))
outer_race = outer_race.cut(outer_groove)

# Create inner race
inner_race_outer = bore_diameter + inner_race_wall * 2 + ball_diameter
inner_race = cq.Workplane("XY").circle(inner_race_outer/2).extrude(width)

# Cut bore
bore_hole = cq.Workplane("XY").circle(bore_diameter/2).extrude(width + 1)
bore_hole = bore_hole.translate((0, 0, -0.5))
inner_race = inner_race.cut(bore_hole)

# Create inner groove
inner_groove_diameter = bore_diameter + inner_race_wall * 2 + ball_diameter/2
inner_groove = cq.Workplane("XY").circle(inner_groove_diameter/2 - groove_depth).extrude(ball_diameter * 0.9)
inner_groove = inner_groove.translate((0, 0, width/2 - ball_diameter * 0.45))
inner_race = inner_race.intersect(inner_groove)

# Create balls
balls_assembly = cq.Workplane("XY")
for i in range(num_balls):
    angle = (360 / num_balls) * i
    x = (pitch_diameter/2) * math.cos(math.radians(angle))
    y = (pitch_diameter/2) * math.sin(math.radians(angle))
    
    ball = cq.Workplane("XY").sphere(ball_diameter/2)
    ball = ball.translate((x, y, width/2))
    
    if i == 0:
        balls_assembly = ball
    else:
        balls_assembly = balls_assembly.union(ball)

# Create simplified ball cage
cage_thickness = 1.2
cage_height = ball_diameter * 0.7

# Outer ring of cage
cage_outer_dia = pitch_diameter + ball_diameter/2 + cage_thickness
cage_outer = cq.Workplane("XY").circle(cage_outer_dia/2).extrude(cage_height)
cage_outer = cage_outer.translate((0, 0, width/2 - cage_height/2))

# Inner ring of cage
cage_inner_dia = pitch_diameter - ball_diameter/2 - cage_thickness
cage_inner = cq.Workplane("XY").circle(cage_inner_dia/2).extrude(cage_height + 1)
cage_inner = cage_inner.translate((0, 0, width/2 - cage_height/2 - 0.5))

cage = cage_outer.cut(cage_inner)

# Cut ball pockets
for i in range(num_balls):
    angle = (360 / num_balls) * i
    x = (pitch_diameter/2) * math.cos(math.radians(angle))
    y = (pitch_diameter/2) * math.sin(math.radians(angle))
    
    pocket = cq.Workplane("XY").circle(ball_diameter/2 + 0.3).extrude(cage_height + 1)
    pocket = pocket.translate((x, y, width/2 - cage_height/2 - 0.5))
    cage = cage.cut(pocket)

# Add cage support ribs between balls
rib_width = 1.5
for i in range(num_balls):
    angle = (360 / num_balls) * i + (360 / num_balls / 2)
    
    # Create rib as radial segment
    rib_inner = cage_inner_dia/2
    rib_outer = cage_outer_dia/2
    half_angle = 3  # degrees
    
    p1 = (rib_inner * math.cos(math.radians(angle - half_angle)), 
          rib_inner * math.sin(math.radians(angle - half_angle)))
    p2 = (rib_outer * math.cos(math.radians(angle - half_angle)), 
          rib_outer * math.sin(math.radians(angle - half_angle)))
    p3 = (rib_outer * math.cos(math.radians(angle + half_angle)), 
          rib_outer * math.sin(math.radians(angle + half_angle)))
    p4 = (rib_inner * math.cos(math.radians(angle + half_angle)), 
          rib_inner * math.sin(math.radians(angle + half_angle)))
    
    rib = cq.Workplane("XY").moveTo(p1[0], p1[1])
    rib = rib.lineTo(p2[0], p2[1])
    rib = rib.lineTo(p3[0], p3[1])
    rib = rib.lineTo(p4[0], p4[1])
    rib = rib.close().extrude(cage_height)
    rib = rib.translate((0, 0, width/2 - cage_height/2))
    
    cage = cage.union(rib)

# Combine all parts
bearing = outer_race.union(inner_race).union(balls_assembly).union(cage)

# Add oil/grease holes in outer race
oil_hole_diameter = 2
num_oil_holes = 3

for i in range(num_oil_holes):
    angle = (360 / num_oil_holes) * i
    x = (outer_diameter/2 - outer_race_wall/2) * math.cos(math.radians(angle))
    y = (outer_diameter/2 - outer_race_wall/2) * math.sin(math.radians(angle))
    
    oil_hole = cq.Workplane("XY").circle(oil_hole_diameter/2).extrude(outer_race_wall + 1)
    oil_hole = oil_hole.rotate((0, 0, 0), (0, 1, 0), 90)
    oil_hole = oil_hole.rotate((0, 0, 0), (0, 0, 1), angle)
    oil_hole = oil_hole.translate((x * 0.7, y * 0.7, width/2))
    bearing = bearing.cut(oil_hole)

show(bearing)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [26]:
#23
import cadquery as cq
from jupyter_cadquery import show
import math

# Flanged Pipe Fitting - Used for connecting pipes in industrial systems
# Simplified reliable version

# Pipe parameters
pipe_outer_diameter = 60
pipe_inner_diameter = 50
pipe_length = 80

# Flange parameters
flange_diameter = 120
flange_thickness = 15
raised_face_diameter = 80
raised_face_height = 2

# Bolt hole parameters
bolt_hole_diameter = 12
bolt_circle_diameter = 95
num_bolt_holes = 8

# Gasket groove parameters
groove_diameter = 70
groove_width = 3
groove_depth = 1.5

# Create main pipe body
pipe_outer = cq.Workplane("XY").circle(pipe_outer_diameter/2).extrude(pipe_length)
pipe_inner = cq.Workplane("XY").circle(pipe_inner_diameter/2).extrude(pipe_length + 1)
pipe_inner = pipe_inner.translate((0, 0, -0.5))
pipe = pipe_outer.cut(pipe_inner)

# Create flange on one end
flange = cq.Workplane("XY").circle(flange_diameter/2).extrude(flange_thickness)
flange = flange.translate((0, 0, pipe_length))

# Combine pipe and flange
fitting = pipe.union(flange)

# Create raised face on flange
raised_face = cq.Workplane("XY").circle(raised_face_diameter/2).extrude(raised_face_height)
raised_face = raised_face.translate((0, 0, pipe_length + flange_thickness))
fitting = fitting.union(raised_face)

# Create bolt holes in flange
for i in range(num_bolt_holes):
    angle = (360 / num_bolt_holes) * i
    x = (bolt_circle_diameter/2) * math.cos(math.radians(angle))
    y = (bolt_circle_diameter/2) * math.sin(math.radians(angle))
    
    # Through hole
    bolt_hole = cq.Workplane("XY").circle(bolt_hole_diameter/2).extrude(flange_thickness + 1)
    bolt_hole = bolt_hole.translate((x, y, pipe_length - 0.5))
    fitting = fitting.cut(bolt_hole)
    
    # Counterbore for bolt heads
    counterbore_diameter = bolt_hole_diameter * 1.8
    counterbore_depth = 8
    counterbore = cq.Workplane("XY").circle(counterbore_diameter/2).extrude(counterbore_depth)
    counterbore = counterbore.translate((x, y, pipe_length - 0.5))
    fitting = fitting.cut(counterbore)

# Create gasket groove
groove_outer = cq.Workplane("XY").circle(groove_diameter/2 + groove_width/2).extrude(groove_depth)
groove_outer = groove_outer.translate((0, 0, pipe_length + flange_thickness + raised_face_height - groove_depth))

groove_inner = cq.Workplane("XY").circle(groove_diameter/2 - groove_width/2).extrude(groove_depth + 1)
groove_inner = groove_inner.translate((0, 0, pipe_length + flange_thickness + raised_face_height - groove_depth - 0.5))

groove = groove_outer.cut(groove_inner)
fitting = fitting.cut(groove)

# Add taper on pipe inlet for easier insertion
inlet_taper_length = 10
taper_scale = 1.15

taper_cutter = cq.Workplane("XY").circle(pipe_inner_diameter/2 * taper_scale).extrude(inlet_taper_length)
taper_cutter = taper_cutter.translate((0, 0, -inlet_taper_length))

# Create cone shape for taper
for i in range(10):
    z = -inlet_taper_length + i * (inlet_taper_length / 10)
    scale = 1 + (taper_scale - 1) * (1 - i/10)
    ring = cq.Workplane("XY").circle(pipe_inner_diameter/2 * scale).extrude(inlet_taper_length / 10 + 0.5)
    ring = ring.translate((0, 0, z))
    fitting = fitting.cut(ring)

# Add reinforcement ribs on pipe exterior
rib_width = 4
rib_height = 8
num_ribs = 6

for i in range(num_ribs):
    angle = (360 / num_ribs) * i
    
    # Create rib as a box
    rib_length = pipe_outer_diameter/2 + rib_height
    rib = cq.Workplane("XY").box(rib_width, rib_length, pipe_length * 0.6)
    rib = rib.translate((0, rib_length/2 - rib_height, pipe_length * 0.5))
    rib = rib.rotate((0, 0, 0), (0, 0, 1), angle)
    
    # Intersect with cylinder to create rib on surface
    rib_mask = cq.Workplane("XY").circle(pipe_outer_diameter/2 + rib_height).extrude(pipe_length * 0.6)
    rib_mask = rib_mask.translate((0, 0, pipe_length * 0.2))
    rib = rib.intersect(rib_mask)
    
    # Remove pipe interior from rib
    rib_hole = cq.Workplane("XY").circle(pipe_outer_diameter/2 - 0.5).extrude(pipe_length * 0.6 + 1)
    rib_hole = rib_hole.translate((0, 0, pipe_length * 0.2 - 0.5))
    rib = rib.cut(rib_hole)
    
    fitting = fitting.union(rib)

# Add drain hole at bottom of pipe
drain_hole_diameter = 6
drain_depth = 15
drain_hole = cq.Workplane("XY").circle(drain_hole_diameter/2).extrude(drain_depth)
drain_hole = drain_hole.rotate((0, 0, 0), (1, 0, 0), 90)
drain_hole = drain_hole.translate((0, -pipe_outer_diameter/2 - 1, pipe_length/2))
fitting = fitting.cut(drain_hole)

# Add pressure relief holes
relief_hole_diameter = 3
num_relief_holes = 4

for i in range(num_relief_holes):
    angle = (360 / num_relief_holes) * i + 45
    x = (pipe_outer_diameter/2 + 2) * math.cos(math.radians(angle))
    y = (pipe_outer_diameter/2 + 2) * math.sin(math.radians(angle))
    
    relief_hole = cq.Workplane("XY").circle(relief_hole_diameter/2).extrude(5)
    relief_hole = relief_hole.rotate((0, 0, 0), (0, 1, 0), 90)
    relief_hole = relief_hole.rotate((0, 0, 0), (0, 0, 1), angle)
    relief_hole = relief_hole.translate((x * 0.7, y * 0.7, pipe_length * 0.75))
    fitting = fitting.cut(relief_hole)

show(fitting)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [27]:
#24
import cadquery as cq
from jupyter_cadquery import show
import math

# Timing Belt Pulley - Used in synchronous drive systems
# Features trapezoidal teeth for belt engagement

# Pulley parameters
pulley_diameter = 60
pulley_width = 20
hub_diameter = 30
hub_length = 25
bore_diameter = 16

# Tooth parameters
num_teeth = 30
tooth_depth = 3
tooth_width_top = 3
tooth_width_bottom = 4
tooth_pitch = (pulley_diameter * math.pi) / num_teeth

# Flange parameters
flange_diameter = pulley_diameter + 8
flange_thickness = 2

# Create main pulley body
pulley_body = cq.Workplane("XY").circle(pulley_diameter/2).extrude(pulley_width)

# Create hub
hub = cq.Workplane("XY").circle(hub_diameter/2).extrude(hub_length)
hub = hub.translate((0, 0, pulley_width))
pulley = pulley_body.union(hub)

# Create bore
bore = cq.Workplane("XY").circle(bore_diameter/2).extrude(pulley_width + hub_length + 1)
bore = bore.translate((0, 0, -0.5))
pulley = pulley.cut(bore)

# Create keyway
keyway_width = 5
keyway_depth = 2.5
keyway = cq.Workplane("XY").box(keyway_width, bore_diameter + 2, pulley_width + hub_length + 1)
keyway = keyway.translate((0, 0, (pulley_width + hub_length)/2 - 0.5))
pulley = pulley.cut(keyway)

# Create timing belt teeth around circumference
tooth_base_radius = pulley_diameter/2 - tooth_depth
tooth_top_radius = pulley_diameter/2

for i in range(num_teeth):
    angle = (360 / num_teeth) * i
    
    # Calculate tooth profile points (trapezoidal)
    angle_rad = math.radians(angle)
    half_top_angle = math.degrees(math.atan(tooth_width_top/2 / tooth_top_radius))
    half_bottom_angle = math.degrees(math.atan(tooth_width_bottom/2 / tooth_base_radius))
    
    # Top of tooth (outer)
    p1_x = tooth_top_radius * math.cos(math.radians(angle - half_top_angle))
    p1_y = tooth_top_radius * math.sin(math.radians(angle - half_top_angle))
    
    p2_x = tooth_top_radius * math.cos(math.radians(angle + half_top_angle))
    p2_y = tooth_top_radius * math.sin(math.radians(angle + half_top_angle))
    
    # Bottom of tooth (inner)
    p3_x = tooth_base_radius * math.cos(math.radians(angle + half_bottom_angle))
    p3_y = tooth_base_radius * math.sin(math.radians(angle + half_bottom_angle))
    
    p4_x = tooth_base_radius * math.cos(math.radians(angle - half_bottom_angle))
    p4_y = tooth_base_radius * math.sin(math.radians(angle - half_bottom_angle))
    
    # Create tooth
    tooth = cq.Workplane("XY").moveTo(p1_x, p1_y)
    tooth = tooth.lineTo(p2_x, p2_y)
    tooth = tooth.lineTo(p3_x, p3_y)
    tooth = tooth.lineTo(p4_x, p4_y)
    tooth = tooth.close().extrude(pulley_width)
    
    pulley = pulley.union(tooth)

# Create flanges on both sides
flange_top = cq.Workplane("XY").circle(flange_diameter/2).extrude(flange_thickness)
flange_top = flange_top.translate((0, 0, pulley_width))
pulley = pulley.union(flange_top)

flange_bottom = cq.Workplane("XY").circle(flange_diameter/2).extrude(flange_thickness)
flange_bottom = flange_bottom.translate((0, 0, -flange_thickness))
pulley = pulley.union(flange_bottom)

# Add lightening holes in hub
lightening_hole_diameter = 6
lightening_circle_diameter = (bore_diameter + hub_diameter) / 2
num_lightening_holes = 4

for i in range(num_lightening_holes):
    angle = (360 / num_lightening_holes) * i
    x = (lightening_circle_diameter/2) * math.cos(math.radians(angle))
    y = (lightening_circle_diameter/2) * math.sin(math.radians(angle))
    
    hole = cq.Workplane("XY").circle(lightening_hole_diameter/2).extrude(hub_length + 1)
    hole = hole.translate((x, y, pulley_width - 0.5))
    pulley = pulley.cut(hole)

# Add set screw holes in hub
set_screw_diameter = 4
num_set_screws = 2

for i in range(num_set_screws):
    angle = (360 / num_set_screws) * i
    x = (hub_diameter/2 - 1) * math.cos(math.radians(angle))
    y = (hub_diameter/2 - 1) * math.sin(math.radians(angle))
    
    screw_hole = cq.Workplane("XY").circle(set_screw_diameter/2).extrude(hub_diameter/2 + 2)
    screw_hole = screw_hole.rotate((0, 0, 0), (0, 1, 0), 90)
    screw_hole = screw_hole.rotate((0, 0, 0), (0, 0, 1), angle)
    screw_hole = screw_hole.translate((x * 0.4, y * 0.4, pulley_width + hub_length/2))
    pulley = pulley.cut(screw_hole)

show(pulley)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [28]:
#25
import cadquery as cq
from jupyter_cadquery import show
import math

# Decorative Candle Holder - Ornamental design with light patterns
# Features geometric cutouts and decorative base

# Main body parameters
body_diameter_bottom = 80
body_diameter_top = 60
body_height = 100
wall_thickness = 3

# Candle well parameters
candle_diameter = 40
candle_depth = 25

# Base parameters
base_diameter = 100
base_height = 15

# Decorative rim parameters
rim_height = 8
rim_diameter = body_diameter_top + 6

# Create tapered main body
body_outer = cq.Workplane("XY").circle(body_diameter_bottom/2).workplane(offset=body_height).circle(body_diameter_top/2).loft()

body_inner = cq.Workplane("XY").circle(body_diameter_bottom/2 - wall_thickness).workplane(offset=body_height).circle(body_diameter_top/2 - wall_thickness).loft()
body_inner = body_inner.translate((0, 0, base_height))

body = body_outer.cut(body_inner)

# Create base platform
base = cq.Workplane("XY").circle(base_diameter/2).extrude(base_height)

# Combine body and base
candle_holder = base.union(body.translate((0, 0, base_height)))

# Create candle well at top
candle_well = cq.Workplane("XY").circle(candle_diameter/2).extrude(candle_depth)
candle_well = candle_well.translate((0, 0, base_height + body_height - candle_depth))
candle_holder = candle_holder.cut(candle_well)

# Create decorative rim at top
rim = cq.Workplane("XY").circle(rim_diameter/2).extrude(rim_height)
rim = rim.translate((0, 0, base_height + body_height))

rim_inner = cq.Workplane("XY").circle(body_diameter_top/2 - wall_thickness/2).extrude(rim_height + 1)
rim_inner = rim_inner.translate((0, 0, base_height + body_height - 0.5))
rim = rim.cut(rim_inner)

candle_holder = candle_holder.union(rim)

# Add geometric cutout pattern - stars/diamonds
num_pattern_rings = 3
cutout_width = 8
cutout_height = 20

for ring_idx in range(num_pattern_rings):
    z_position = base_height + 20 + ring_idx * 25
    num_cutouts = 8
    
    # Alternate rotation for each ring
    rotation_offset = (360 / num_cutouts / 2) * (ring_idx % 2)
    
    for i in range(num_cutouts):
        angle = (360 / num_cutouts) * i + rotation_offset
        
        # Calculate radius at this height
        height_ratio = (z_position - base_height) / body_height
        radius_at_height = body_diameter_bottom/2 - (body_diameter_bottom/2 - body_diameter_top/2) * height_ratio
        
        x = radius_at_height * math.cos(math.radians(angle))
        y = radius_at_height * math.sin(math.radians(angle))
        
        # Diamond-shaped cutout
        cutout = cq.Workplane("XY").box(cutout_width, wall_thickness + 2, cutout_height)
        cutout = cutout.rotate((0, 0, 0), (0, 0, 1), 45)  # Rotate to make diamond
        cutout = cutout.rotate((0, 0, 0), (0, 0, 1), angle)
        cutout = cutout.translate((x, y, z_position))
        
        candle_holder = candle_holder.cut(cutout)

# Add decorative scalloped edge to base
num_scallops = 12
scallop_depth = 4

for i in range(num_scallops):
    angle = (360 / num_scallops) * i
    x = (base_diameter/2 - scallop_depth/2) * math.cos(math.radians(angle))
    y = (base_diameter/2 - scallop_depth/2) * math.sin(math.radians(angle))
    
    scallop = cq.Workplane("XY").circle(scallop_depth).extrude(base_height + 1)
    scallop = scallop.translate((x, y, -0.5))
    candle_holder = candle_holder.cut(scallop)

# Add vertical accent grooves on body
num_grooves = 16
groove_width = 2
groove_depth = 1.5

for i in range(num_grooves):
    angle = (360 / num_grooves) * i
    
    # Create vertical groove
    groove_points = []
    num_points = 10
    for j in range(num_points):
        z = base_height + (body_height / num_points) * j
        height_ratio = j / num_points
        radius = body_diameter_bottom/2 - (body_diameter_bottom/2 - body_diameter_top/2) * height_ratio - groove_depth
        
        x = radius * math.cos(math.radians(angle))
        y = radius * math.sin(math.radians(angle))
        
        groove_segment = cq.Workplane("XY").box(groove_width, wall_thickness + 2, body_height / num_points + 0.5)
        groove_segment = groove_segment.rotate((0, 0, 0), (0, 0, 1), angle)
        groove_segment = groove_segment.translate((x, y, z))
        candle_holder = candle_holder.cut(groove_segment)

# Add decorative band around middle
band_height = 8
band_depth = 2
band_z = base_height + body_height/2

for i in range(24):
    angle = (360 / 24) * i
    height_ratio = 0.5
    radius = body_diameter_bottom/2 - (body_diameter_bottom/2 - body_diameter_top/2) * height_ratio
    
    x = (radius - band_depth/2) * math.cos(math.radians(angle))
    y = (radius - band_depth/2) * math.sin(math.radians(angle))
    
    # Small rectangular indent
    indent = cq.Workplane("XY").box(3, wall_thickness + 2, band_height)
    indent = indent.rotate((0, 0, 0), (0, 0, 1), angle)
    indent = indent.translate((x, y, band_z))
    candle_holder = candle_holder.cut(indent)

# Add drain hole in candle well (for wax)
drain_hole = cq.Workplane("XY").circle(2).extrude(base_height + body_height - candle_depth + 1)
drain_hole = drain_hole.translate((candle_diameter/3, 0, base_height + body_height - candle_depth - 0.5))
candle_holder = candle_holder.cut(drain_hole)

show(candle_holder)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [29]:
#26
import cadquery as cq
from jupyter_cadquery import show
import math

# Ornate Picture Frame - Decorative frame with geometric relief
# Simplified reliable version

# Frame parameters
frame_outer_width = 200
frame_outer_height = 250
frame_depth = 30
frame_border_width = 35

# Picture opening parameters
picture_width = frame_outer_width - 2 * frame_border_width
picture_height = frame_outer_height - 2 * frame_border_width
picture_recess_depth = 15

# Relief pattern parameters
relief_depth = 5
bead_diameter = 8

# Create main frame body
frame_outer = cq.Workplane("XY").box(frame_outer_width, frame_outer_height, frame_depth)

# Cut picture opening
picture_opening = cq.Workplane("XY").box(picture_width, picture_height, picture_recess_depth + 1)
picture_opening = picture_opening.translate((0, 0, frame_depth/2 - picture_recess_depth/2 + 0.5))
frame = frame_outer.cut(picture_opening)

# Create picture rabbet (ledge for glass/picture)
rabbet_width = 5
rabbet_depth = 3
rabbet_size_w = picture_width + rabbet_width*2
rabbet_size_h = picture_height + rabbet_width*2

rabbet_cut = cq.Workplane("XY").box(rabbet_size_w, rabbet_size_h, rabbet_depth + 1)
rabbet_cut = rabbet_cut.translate((0, 0, frame_depth/2 - picture_recess_depth - rabbet_depth/2 - 0.5))

inner_cut = cq.Workplane("XY").box(picture_width, picture_height, rabbet_depth + 2)
inner_cut = inner_cut.translate((0, 0, frame_depth/2 - picture_recess_depth - rabbet_depth/2 - 0.5))

frame = frame.cut(rabbet_cut)
frame = frame.union(rabbet_cut.cut(inner_cut).translate((0, 0, rabbet_depth/2)))

# Add decorative raised border strips
strip_width = 8
strip_height = 3
strip_offset = 15

# Top strip
top_strip = cq.Workplane("XY").box(frame_outer_width - strip_offset*2, strip_width, strip_height)
top_strip = top_strip.translate((0, frame_outer_height/2 - frame_border_width/2, frame_depth/2 + strip_height/2))
frame = frame.union(top_strip)

# Bottom strip
bottom_strip = cq.Workplane("XY").box(frame_outer_width - strip_offset*2, strip_width, strip_height)
bottom_strip = bottom_strip.translate((0, -frame_outer_height/2 + frame_border_width/2, frame_depth/2 + strip_height/2))
frame = frame.union(bottom_strip)

# Left strip
left_strip = cq.Workplane("XY").box(strip_width, frame_outer_height - strip_offset*2, strip_height)
left_strip = left_strip.translate((-frame_outer_width/2 + frame_border_width/2, 0, frame_depth/2 + strip_height/2))
frame = frame.union(left_strip)

# Right strip
right_strip = cq.Workplane("XY").box(strip_width, frame_outer_height - strip_offset*2, strip_height)
right_strip = right_strip.translate((frame_outer_width/2 - frame_border_width/2, 0, frame_depth/2 + strip_height/2))
frame = frame.union(right_strip)

# Add corner ornaments (decorative bosses)
corner_size = 20
corner_height = 8
corner_offset = frame_border_width/2

for x_sign in [-1, 1]:
    for y_sign in [-1, 1]:
        x_pos = x_sign * (frame_outer_width/2 - corner_offset)
        y_pos = y_sign * (frame_outer_height/2 - corner_offset)
        
        # Create decorative corner piece
        corner = cq.Workplane("XY").circle(corner_size/2).extrude(corner_height)
        
        # Add smaller circle on top
        corner_top = cq.Workplane("XY").circle(corner_size/3).extrude(corner_height/2)
        corner_top = corner_top.translate((0, 0, corner_height))
        corner = corner.union(corner_top)
        
        corner = corner.translate((x_pos, y_pos, frame_depth/2))
        frame = frame.union(corner)

# Add beaded edge pattern around inner border
num_beads_horizontal = 12
num_beads_vertical = 15

bead_offset = frame_border_width * 0.4
bead_protrusion = 3

# Top edge beads
for i in range(num_beads_horizontal):
    x = -picture_width/2 + (picture_width / (num_beads_horizontal - 1)) * i
    y = picture_height/2 + bead_offset
    
    bead = cq.Workplane("XY").sphere(bead_diameter/2)
    bead = bead.translate((x, y, frame_depth/2 + bead_protrusion))
    frame = frame.union(bead)

# Bottom edge beads
for i in range(num_beads_horizontal):
    x = -picture_width/2 + (picture_width / (num_beads_horizontal - 1)) * i
    y = -picture_height/2 - bead_offset
    
    bead = cq.Workplane("XY").sphere(bead_diameter/2)
    bead = bead.translate((x, y, frame_depth/2 + bead_protrusion))
    frame = frame.union(bead)

# Left edge beads
for i in range(num_beads_vertical):
    x = -picture_width/2 - bead_offset
    y = -picture_height/2 + (picture_height / (num_beads_vertical - 1)) * i
    
    bead = cq.Workplane("XY").sphere(bead_diameter/2)
    bead = bead.translate((x, y, frame_depth/2 + bead_protrusion))
    frame = frame.union(bead)

# Right edge beads
for i in range(num_beads_vertical):
    x = picture_width/2 + bead_offset
    y = -picture_height/2 + (picture_height / (num_beads_vertical - 1)) * i
    
    bead = cq.Workplane("XY").sphere(bead_diameter/2)
    bead = bead.translate((x, y, frame_depth/2 + bead_protrusion))
    frame = frame.union(bead)

# Add diagonal decorative grooves in corners
groove_length = 30
groove_width = 3
groove_depth = 2

for corner_angle in [45, 135, 225, 315]:
    for offset in [-10, 0, 10]:
        x_center = (frame_outer_width/2 - frame_border_width/2) * math.cos(math.radians(corner_angle))
        y_center = (frame_outer_height/2 - frame_border_width/2) * math.sin(math.radians(corner_angle))
        
        # Offset perpendicular to diagonal
        perp_angle = corner_angle + 90
        x = x_center + offset * math.cos(math.radians(perp_angle))
        y = y_center + offset * math.sin(math.radians(perp_angle))
        
        groove = cq.Workplane("XY").box(groove_width, groove_length, groove_depth)
        groove = groove.rotate((0, 0, 0), (0, 0, 1), corner_angle)
        groove = groove.translate((x, y, frame_depth/2 + groove_depth/2))
        frame = frame.cut(groove)

# Add hanging hardware holes on back
hanging_hole_diameter = 8
hanging_hole_depth = 15

for x_sign in [-1, 1]:
    x_pos = x_sign * (frame_outer_width/2 - frame_border_width)
    
    hanging_hole = cq.Workplane("XY").circle(hanging_hole_diameter/2).extrude(hanging_hole_depth)
    hanging_hole = hanging_hole.translate((x_pos, frame_outer_height/2 - frame_border_width/2, -frame_depth/2 - 0.5))
    frame = frame.cut(hanging_hole)

# Add decorative holes pattern on border
hole_diameter = 4
num_holes_per_side = 8

# Top border holes
for i in range(num_holes_per_side):
    x = -frame_outer_width/2 + frame_border_width + (frame_outer_width - 2*frame_border_width) * i / (num_holes_per_side - 1)
    y = frame_outer_height/2 - frame_border_width/2
    
    hole = cq.Workplane("XY").circle(hole_diameter/2).extrude(relief_depth + 1)
    hole = hole.translate((x, y, frame_depth/2 - 0.5))
    frame = frame.cut(hole)

# Bottom border holes
for i in range(num_holes_per_side):
    x = -frame_outer_width/2 + frame_border_width + (frame_outer_width - 2*frame_border_width) * i / (num_holes_per_side - 1)
    y = -frame_outer_height/2 + frame_border_width/2
    
    hole = cq.Workplane("XY").circle(hole_diameter/2).extrude(relief_depth + 1)
    hole = hole.translate((x, y, frame_depth/2 - 0.5))
    frame = frame.cut(hole)

# Side border holes
num_holes_vertical = 10
for i in range(num_holes_vertical):
    y = -frame_outer_height/2 + frame_border_width + (frame_outer_height - 2*frame_border_width) * i / (num_holes_vertical - 1)
    
    # Left holes
    hole_left = cq.Workplane("XY").circle(hole_diameter/2).extrude(relief_depth + 1)
    hole_left = hole_left.translate((-frame_outer_width/2 + frame_border_width/2, y, frame_depth/2 - 0.5))
    frame = frame.cut(hole_left)
    
    # Right holes
    hole_right = cq.Workplane("XY").circle(hole_diameter/2).extrude(relief_depth + 1)
    hole_right = hole_right.translate((frame_outer_width/2 - frame_border_width/2, y, frame_depth/2 - 0.5))
    frame = frame.cut(hole_right)

show(frame)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [30]:
#27
import cadquery as cq
from jupyter_cadquery import show
import math

# Celtic Knot Coaster - Interwoven endless knot pattern with raised rim
# Parameters
coaster_diameter = 90.0     # Overall coaster diameter
coaster_thickness = 8.0     # Total thickness
rim_width = 5.0            # Width of outer rim
rim_height = 2.0           # Raised rim height
base_thickness = 3.0       # Base layer thickness

# Knot pattern parameters
knot_width = 6.0           # Width of knot strands
knot_height = 3.0          # Height of raised knot
weave_radius = 25.0        # Radius of main weave circle
num_loops = 3              # Number of loops in pattern
over_under_gap = 0.8       # Gap for over/under weave effect

# Cork feet parameters
foot_radius = 8.0          # Radius of cork feet
foot_depth = 1.0           # Recess depth for cork
num_feet = 4               # Number of feet

# Drainage parameters
channel_width = 2.0        # Width of drainage channels
channel_depth = 1.0        # Depth of drainage channels

# Create base disc with raised rim
model = (cq.Workplane("XY")
    .circle(coaster_diameter/2)
    .extrude(base_thickness)
)

# Add raised rim
rim = (cq.Workplane("XY")
    .workplane(offset=base_thickness)
    .circle(coaster_diameter/2)
    .circle(coaster_diameter/2 - rim_width)
    .extrude(rim_height)
)
model = model.union(rim)

# Add decorative chamfer to rim
model = model.edges(">Z").chamfer(0.5)

# Create Celtic trinity knot pattern using mathematical curves
for loop in range(num_loops):
    base_angle = loop * 120  # Three-fold symmetry
    
    # Create main loop path using parametric equations
    points = []
    for t in range(60):
        angle = t * 6  # 0 to 360 degrees
        
        # Parametric equations for a three-lobed curve
        r = weave_radius * (1 + 0.3 * math.cos(3 * math.radians(angle)))
        x = r * math.cos(math.radians(angle + base_angle))
        y = r * math.sin(math.radians(angle + base_angle))
        points.append((x, y))
    
    # Create knot strand from points
    for i in range(len(points)):
        next_i = (i + 1) % len(points)
        
        # Calculate perpendicular direction for strand width
        dx = points[next_i][0] - points[i][0]
        dy = points[next_i][1] - points[i][1]
        length = math.sqrt(dx*dx + dy*dy) if dx != 0 or dy != 0 else 0.1
        
        if length > 0.1:  # Skip very short segments
            # Create strand segment
            strand = (cq.Workplane("XY")
                .workplane(offset=base_thickness)
                .center(points[i][0], points[i][1])
                .box(knot_width, length * 1.2, knot_height)
                .rotate((0, 0, 0), (0, 0, 1), math.degrees(math.atan2(dy, dx)) + 90)
            )
            
            # Determine over/under pattern
            if (i // 10) % 2 == loop % 2:  # Creates interwoven effect
                model = model.union(strand)
            else:
                # Create "under" sections with gaps
                strand_under = strand.translate((0, 0, -over_under_gap))
                model = model.union(strand_under)

# Add center medallion
medallion = (cq.Workplane("XY")
    .workplane(offset=base_thickness)
    .circle(12)
    .extrude(2)
)

# Create triquetra symbol in center
for i in range(3):
    angle = i * 120
    petal_x = 6 * math.cos(math.radians(angle))
    petal_y = 6 * math.sin(math.radians(angle))
    
    petal = (cq.Workplane("XY")
        .workplane(offset=base_thickness + 2)
        .center(petal_x, petal_y)
        .circle(4)
        .extrude(-1.5)
    )
    medallion = medallion.cut(petal)

model = model.union(medallion)

# Add drainage channels in cross pattern
for angle in [0, 90]:
    channel = (cq.Workplane("XY")
        .workplane(offset=base_thickness - channel_depth)
        .box(coaster_diameter - rim_width*2, channel_width, channel_depth)
        .rotate((0, 0, 0), (0, 0, 1), angle)
    )
    model = model.cut(channel)

# Add recessed areas for cork feet
foot_positions = []
for i in range(num_feet):
    angle = i * 90 + 45  # 45° offset for square pattern
    foot_x = (coaster_diameter/2 - 15) * math.cos(math.radians(angle))
    foot_y = (coaster_diameter/2 - 15) * math.sin(math.radians(angle))
    
    foot_recess = (cq.Workplane("XY")
        .center(foot_x, foot_y)
        .circle(foot_radius)
        .extrude(foot_depth)
    )
    model = model.cut(foot_recess)
    
    # Add grip pattern around feet
    for j in range(6):
        grip_angle = j * 60
        grip_x = foot_x + (foot_radius + 3) * math.cos(math.radians(grip_angle))
        grip_y = foot_y + (foot_radius + 3) * math.sin(math.radians(grip_angle))
        
        grip_dot = (cq.Workplane("XY")
            .center(grip_x, grip_y)
            .circle(1)
            .extrude(0.5)
        )
        model = model.cut(grip_dot)

# Add decorative border pattern
for i in range(24):
    angle = i * 15
    border_x = (coaster_diameter/2 - rim_width/2) * math.cos(math.radians(angle))
    border_y = (coaster_diameter/2 - rim_width/2) * math.sin(math.radians(angle))
    
    notch = (cq.Workplane("XY")
        .workplane(offset=base_thickness + rim_height - 0.5)
        .center(border_x, border_y)
        .sphere(1.5)
    )
    model = model.cut(notch)

show(model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [31]:
#28
import cadquery as cq
from jupyter_cadquery import show
import math

# Mechanical Combination Lock Dial - Precision security mechanism with detents
# Parameters
dial_diameter = 85.0        # Main dial diameter
dial_thickness = 15.0       # Total dial thickness
center_hub_dia = 30.0       # Center rotating hub diameter
hub_height = 20.0           # Hub extension height

# Number ring parameters
ring_thickness = 3.0        # Thickness of number ring
ring_height = 8.0           # Height of raised ring
num_count = 40              # Numbers around dial (0-39)
tick_length = 4.0           # Length of tick marks
tick_width = 1.0            # Width of tick marks

# Knurling parameters
knurl_depth = 1.0           # Depth of grip knurls
knurl_count = 60            # Number of knurls around edge
knurl_width = 2.0           # Width of each knurl

# Detent parameters
detent_count = 40           # Number of detent positions
detent_radius = 0.8         # Size of detent notches
detent_ring_radius = 35.0   # Radius of detent ring

# Internal mechanism parameters
gate_width = 5.0            # Width of gate notch
gate_depth = 3.0            # Depth of gate notch
cam_thickness = 5.0         # Thickness of cam disc

# Create main dial body with stepped profile
model = (cq.Workplane("XY")
    .circle(dial_diameter/2)
    .extrude(dial_thickness/3)
)

# Add middle section
middle = (cq.Workplane("XY")
    .workplane(offset=dial_thickness/3)
    .circle(dial_diameter/2 - 2)
    .extrude(dial_thickness/3)
)
model = model.union(middle)

# Add top section with number ring
top = (cq.Workplane("XY")
    .workplane(offset=2*dial_thickness/3)
    .circle(dial_diameter/2 - 4)
    .extrude(dial_thickness/3)
)
model = model.union(top)

# Create raised number ring
number_ring = (cq.Workplane("XY")
    .workplane(offset=dial_thickness)
    .circle(dial_diameter/2 - 4)
    .circle(dial_diameter/2 - 4 - ring_thickness)
    .extrude(ring_height)
)
model = model.union(number_ring)

# Add tick marks and number positions
for i in range(num_count):
    angle = 360 * i / num_count
    rad_angle = math.radians(angle)
    
    # Major tick marks every 5 numbers
    if i % 5 == 0:
        tick_r = dial_diameter/2 - 4 - ring_thickness/2
        tick = (cq.Workplane("XY")
            .workplane(offset=dial_thickness + ring_height - 1)
            .center(tick_r * math.cos(rad_angle), tick_r * math.sin(rad_angle))
            .box(tick_length * 1.5, tick_width * 1.5, 2)
            .rotate((0, 0, 0), (0, 0, 1), angle)
        )
        model = model.cut(tick)
        
        # Add number marker (simplified as deeper notch)
        number_mark = (cq.Workplane("XY")
            .workplane(offset=dial_thickness + ring_height - 2)
            .center((tick_r - 3) * math.cos(rad_angle), (tick_r - 3) * math.sin(rad_angle))
            .circle(1.5)
            .extrude(2)
        )
        model = model.cut(number_mark)
    else:
        # Minor tick marks
        tick_r = dial_diameter/2 - 4 - ring_thickness/2
        tick = (cq.Workplane("XY")
            .workplane(offset=dial_thickness + ring_height - 0.5)
            .center(tick_r * math.cos(rad_angle), tick_r * math.sin(rad_angle))
            .box(tick_length, tick_width, 1)
            .rotate((0, 0, 0), (0, 0, 1), angle)
        )
        model = model.cut(tick)

# Add center hub with shaft hole
hub = (cq.Workplane("XY")
    .workplane(offset=dial_thickness)
    .circle(center_hub_dia/2)
    .extrude(hub_height - dial_thickness)
)

# Cut shaft hole through center
shaft_hole = (cq.Workplane("XY")
    .circle(8)  # 8mm shaft
    .extrude(hub_height + 1)
)
hub = hub.cut(shaft_hole)

# Add keyway in shaft
keyway = (cq.Workplane("XY")
    .center(0, 6)
    .box(3, 5, hub_height + 1)
)
hub = hub.cut(keyway)

model = model.union(hub)

# Add knurling around edge for grip
for i in range(knurl_count):
    angle = 360 * i / knurl_count
    rad_angle = math.radians(angle)
    
    knurl_x = (dial_diameter/2 - knurl_depth/2) * math.cos(rad_angle)
    knurl_y = (dial_diameter/2 - knurl_depth/2) * math.sin(rad_angle)
    
    knurl = (cq.Workplane("XY")
        .workplane(offset=2)
        .center(knurl_x, knurl_y)
        .box(knurl_depth * 2, knurl_width, dial_thickness - 4)
        .rotate((0, 0, 0), (0, 0, 1), angle)
    )
    model = model.cut(knurl)

# Add detent notches on back face
for i in range(detent_count):
    angle = 360 * i / detent_count
    rad_angle = math.radians(angle)
    
    detent_x = detent_ring_radius * math.cos(rad_angle)
    detent_y = detent_ring_radius * math.sin(rad_angle)
    
    detent = (cq.Workplane("XY")
        .center(detent_x, detent_y)
        .sphere(detent_radius)
    )
    model = model.cut(detent)

# Add gate notch for true combination position
gate_angle = 0  # True gate at 0 position
gate_x = (dial_diameter/2 - 10) * math.cos(math.radians(gate_angle))
gate_y = (dial_diameter/2 - 10) * math.sin(math.radians(gate_angle))

gate_notch = (cq.Workplane("XY")
    .workplane(offset=cam_thickness)
    .center(gate_x, gate_y)
    .box(gate_width, gate_depth * 2, cam_thickness)
    .rotate((0, 0, 0), (0, 0, 1), gate_angle)
)
model = model.cut(gate_notch)

# Add false gates at other positions for security
false_gate_positions = [90, 180, 270]
for angle in false_gate_positions:
    rad_angle = math.radians(angle)
    false_x = (dial_diameter/2 - 10) * math.cos(rad_angle)
    false_y = (dial_diameter/2 - 10) * math.sin(rad_angle)
    
    false_gate = (cq.Workplane("XY")
        .workplane(offset=cam_thickness)
        .center(false_x, false_y)
        .box(gate_width * 0.7, gate_depth, cam_thickness * 0.7)
        .rotate((0, 0, 0), (0, 0, 1), angle)
    )
    model = model.cut(false_gate)

# Add reference arrow marker at top
arrow_size = 6.0
arrow = (cq.Workplane("XY")
    .workplane(offset=dial_thickness + ring_height)
    .center(0, dial_diameter/2 - 8)
    .polyline([
        (0, arrow_size),
        (-arrow_size/2, 0),
        (arrow_size/2, 0),
        (0, arrow_size)
    ])
    .close()
    .extrude(-1)
)
model = model.cut(arrow)

# Add concentric circle grooves for aesthetics
for radius in [15, 20]:
    groove = (cq.Workplane("XY")
        .workplane(offset=dial_thickness + ring_height - 0.5)
        .circle(radius)
        .circle(radius - 1)
        .extrude(0.5)
    )
    model = model.cut(groove)

show(model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [32]:
#29
import cadquery as cq
from jupyter_cadquery import show
import math

# Modern Phone Stand - Minimalist design with cable pass-through
# Parameters
base_length = 80.0          # Base plate length
base_width = 70.0           # Base plate width  
base_thickness = 8.0        # Base thickness
base_radius = 10.0          # Corner rounding radius

# Support arm parameters
support_height = 65.0       # Height of back support
support_thickness = 10.0    # Thickness of support
support_angle = 15.0        # Lean-back angle in degrees
lip_height = 12.0           # Height of phone lip
lip_depth = 15.0            # Depth of phone lip

# Cable management
cable_slot_width = 12.0     # Width of cable slot
cable_slot_height = 6.0     # Height of cable slot
cable_groove_width = 4.0    # Width of cable groove

# Grip and aesthetic features
rubber_pad_diameter = 25.0  # Diameter of rubber pad recesses
pad_depth = 1.0             # Depth of pad recesses
vent_width = 5.0            # Width of ventilation slots
vent_spacing = 8.0          # Spacing between vents

# Create base with rounded corners
model = (cq.Workplane("XY")
    .box(base_length, base_width, base_thickness)
    .edges("|Z").fillet(base_radius)
)

# Add weight-reducing pocket underneath
pocket = (cq.Workplane("XY")
    .box(base_length - 20, base_width - 20, base_thickness - 2)
)
model = model.cut(pocket)

# Create angled support back
support_bottom_offset = support_height * math.tan(math.radians(support_angle))
support = (cq.Workplane("XZ")
    .workplane(offset=-base_width/2 + 15)
    .polyline([
        (-base_length/2 + 20, base_thickness),
        (-base_length/2 + 20 + support_bottom_offset, support_height),
        (-base_length/2 + 20 + support_bottom_offset + support_thickness, support_height),
        (-base_length/2 + 20 + support_thickness, base_thickness),
        (-base_length/2 + 20, base_thickness)
    ])
    .close()
    .extrude(base_width - 30)
)
model = model.union(support)

# Create phone lip/ledge
lip = (cq.Workplane("XY")
    .workplane(offset=base_thickness)
    .center(5, 0)
    .box(lip_depth, base_width - 30, lip_height)
)
model = model.union(lip)

# Cut cable pass-through slot
cable_slot = (cq.Workplane("XY")
    .workplane(offset=base_thickness)
    .center(5, 0)
    .box(cable_slot_width, cable_slot_height, lip_height + 1)
)
model = model.cut(cable_slot)

# Add cable groove along base
cable_groove = (cq.Workplane("XY")
    .center(5, 0)
    .box(cable_groove_width, base_width, pad_depth)
)
model = model.cut(cable_groove)

# Add ventilation/weight reduction slots in support
num_vents = 4
for i in range(num_vents):
    vent_y = -15 + i * vent_spacing
    vent = (cq.Workplane("XZ")
        .workplane(offset=vent_y)
        .center(-base_length/2 + 25 + support_bottom_offset/2, support_height/2)
        .box(25, vent_width, support_thickness + 1)
        .rotate((-base_length/2 + 25, 0, 0), (0, 1, 0), -support_angle)
    )
    model = model.cut(vent)

# Add rubber pad recesses (4 corners)
pad_positions = [
    (base_length/2 - 15, base_width/2 - 15),
    (-base_length/2 + 15, base_width/2 - 15),
    (base_length/2 - 15, -base_width/2 + 15),
    (-base_length/2 + 15, -base_width/2 + 15)
]

for x, y in pad_positions:
    pad_recess = (cq.Workplane("XY")
        .center(x, y)
        .circle(rubber_pad_diameter/2)
        .extrude(pad_depth)
    )
    model = model.cut(pad_recess)
    
    # Add grip pattern in recess
    for angle in range(0, 360, 60):
        dot_x = x + 5 * math.cos(math.radians(angle))
        dot_y = y + 5 * math.sin(math.radians(angle))
        dot = (cq.Workplane("XY")
            .center(dot_x, dot_y)
            .circle(1)
            .extrude(pad_depth/2)
        )
        model = model.cut(dot)

# Add logo/branding area (recessed rectangle)
logo_area = (cq.Workplane("XZ")
    .workplane(offset=0)
    .center(-base_length/2 + 35 + support_bottom_offset/2, support_height - 15)
    .box(20, 8, 0.5)
    .rotate((-base_length/2 + 25, 0, 0), (0, 1, 0), -support_angle)
)
model = model.cut(logo_area)

# Round edges for comfortable handling
model = model.edges(">Z").fillet(2)
model = model.edges("<Z").fillet(1)

show(model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [33]:
#30
import cadquery as cq
from jupyter_cadquery import show
import math

# Stackable Storage Container Lid - Commercial food/organization container
# Parameters
lid_length = 120.0          # Lid length (rectangular)
lid_width = 80.0            # Lid width
lid_height = 12.0           # Total lid height
rim_width = 8.0             # Width of sealing rim
rim_depth = 4.0             # Depth of rim that fits into container

# Seal and locking parameters
seal_ridge_height = 1.5     # Height of sealing ridge
seal_ridge_width = 2.0      # Width of sealing ridge
lock_tab_width = 20.0       # Width of locking tabs
lock_tab_count = 2          # Number of tabs per side

# Stacking features
stack_ridge_offset = 15.0   # Offset from edge for stacking ridge
stack_ridge_height = 3.0    # Height of stacking ridge
stack_groove_width = 3.5    # Width of stacking groove

# Ventilation parameters
vent_diameter = 3.0         # Diameter of vent holes
vent_count = 4              # Number of vent holes
vent_cover_size = 8.0       # Size of vent cover area

# Handle/grip parameters
handle_length = 40.0        # Length of handle depression
handle_width = 15.0         # Width of handle depression
handle_depth = 3.0          # Depth of handle depression

# Create main lid body
model = (cq.Workplane("XY")
    .box(lid_length, lid_width, lid_height - rim_depth)
)

# Add rim that inserts into container
rim = (cq.Workplane("XY")
    .workplane(offset=lid_height - rim_depth)
    .box(lid_length - rim_width*2, lid_width - rim_width*2, rim_depth)
)
model = model.union(rim)

# Add sealing ridge around rim
seal_ridge = (cq.Workplane("XY")
    .workplane(offset=lid_height - rim_depth + 1)
    .box(lid_length - rim_width*2 - 4, lid_width - rim_width*2 - 4, seal_ridge_height)
    .box(lid_length - rim_width*2 - 6, lid_width - rim_width*2 - 6, seal_ridge_height)
)
model = model.union(seal_ridge)

# Create stacking ridge on top
stack_ridge = (cq.Workplane("XY")
    .workplane(offset=-lid_height/2 + rim_depth/2)
    .box(lid_length - stack_ridge_offset*2, lid_width - stack_ridge_offset*2, stack_ridge_height)
    .box(lid_length - stack_ridge_offset*2 - stack_groove_width*2, 
         lid_width - stack_ridge_offset*2 - stack_groove_width*2, stack_ridge_height)
)
model = model.union(stack_ridge)

# Add locking tabs on short sides
for y in [-lid_width/2, lid_width/2]:
    for i in range(lock_tab_count):
        x_offset = -lock_tab_width * (lock_tab_count - 1)/2 + i * lock_tab_width
        
        tab = (cq.Workplane("XY")
            .workplane(offset=0)
            .center(x_offset, y)
            .box(lock_tab_width - 2, rim_width + 4, lid_height - rim_depth)
        )
        model = model.union(tab)
        
        # Add grip ridges on tabs
        for j in range(3):
            ridge = (cq.Workplane("XY")
                .workplane(offset=-2 + j * 2)
                .center(x_offset, y)
                .box(lock_tab_width - 4, rim_width + 6, 0.5)
            )
            model = model.union(ridge)

# Cut handle depressions on long sides
for x in [-lid_length/2 + rim_width/2, lid_length/2 - rim_width/2]:
    handle = (cq.Workplane("YZ")
        .workplane(offset=x)
        .center(0, -lid_height/2 + rim_depth/2 + 2)
        .ellipse(handle_length/2, handle_depth)
        .extrude(handle_width)
    )
    model = model.cut(handle)
    
    # Add grip dots in handle area
    for i in range(5):
        dot_y = -15 + i * 7.5
        dot = (cq.Workplane("YZ")
            .workplane(offset=x - handle_width/2 + 2)
            .center(dot_y, -lid_height/2 + rim_depth/2 + 2)
            .circle(1.5)
            .extrude(1)
        )
        model = model.cut(dot)

# Add vented area with moisture control
vent_area = (cq.Workplane("XY")
    .workplane(offset=-lid_height/2 + rim_depth/2 + 1)
    .box(vent_cover_size * 3, vent_cover_size, 1)
)
model = model.union(vent_area)

# Cut vent holes
for i in range(vent_count):
    x = -vent_cover_size + i * vent_cover_size
    vent = (cq.Workplane("XY")
        .center(x, 0)
        .circle(vent_diameter/2)
        .extrude(lid_height)
    )
    model = model.cut(vent)
    
    # Add sliding vent cover track
    track = (cq.Workplane("XY")
        .workplane(offset=-lid_height/2 + rim_depth/2 + 1.5)
        .center(x, 0)
        .box(vent_cover_size + 2, vent_cover_size - 2, 0.5)
    )
    model = model.cut(track)

# Add date dial recess (for food storage dating)
dial_radius = 12.0
dial = (cq.Workplane("XY")
    .workplane(offset=-lid_height/2 + rim_depth/2 + 1)
    .center(lid_length/2 - 20, lid_width/2 - 20)
    .circle(dial_radius)
    .extrude(-1)
)
model = model.cut(dial)

# Add day markers around dial
days = 7
for i in range(days):
    angle = 360 * i / days
    marker_x = (lid_length/2 - 20) + (dial_radius - 3) * math.cos(math.radians(angle))
    marker_y = (lid_width/2 - 20) + (dial_radius - 3) * math.sin(math.radians(angle))
    
    marker = (cq.Workplane("XY")
        .workplane(offset=-lid_height/2 + rim_depth/2)
        .center(marker_x, marker_y)
        .circle(1)
        .extrude(-0.5)
    )
    model = model.cut(marker)

# Add recycling symbol recess
symbol = (cq.Workplane("XY")
    .workplane(offset=-lid_height/2 + rim_depth/2 + 0.5)
    .center(0, -lid_width/2 + 15)
    .box(12, 12, 0.5)
)
model = model.cut(symbol)

# Add corner reinforcements
for x_sign in [-1, 1]:
    for y_sign in [-1, 1]:
        corner_x = x_sign * (lid_length/2 - 10)
        corner_y = y_sign * (lid_width/2 - 10)
        
        reinforce = (cq.Workplane("XY")
            .center(corner_x, corner_y)
            .circle(5)
            .extrude(lid_height - rim_depth)
        )
        model = model.union(reinforce)

show(model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [34]:

#31
import cadquery as cq
from jupyter_cadquery import show
import math

# Impossible Triangle Pen Holder - Penrose triangle optical illusion
# Parameters
triangle_size = 70.0        # Size of the triangle sides
beam_width = 12.0           # Width of each beam
beam_height = 12.0          # Height of each beam
twist_offset = 15.0         # Offset that creates the impossible effect

# Pen hole parameters
hole_diameter = 10.0        # Diameter of pen holes
hole_spacing = 20.0         # Spacing between holes

# Create three beams that form the impossible triangle
# First beam - horizontal bottom
beam1 = (cq.Workplane("XY")
    .center(-triangle_size/2, -triangle_size/(2*math.sqrt(3)))
    .box(triangle_size, beam_width, beam_height)
)

# Second beam - angled left (120 degrees)
beam2 = (cq.Workplane("XY")
    .center(0, triangle_size/math.sqrt(3) - beam_width/2)
    .box(triangle_size, beam_width, beam_height)
    .rotate((0, 0, 0), (0, 0, 1), -60)
    .translate((0, 0, twist_offset))
)

# Third beam - angled right (240 degrees)  
beam3 = (cq.Workplane("XY")
    .center(triangle_size/2, -triangle_size/(2*math.sqrt(3)))
    .box(triangle_size, beam_width, beam_height)
    .rotate((0, 0, 0), (0, 0, 1), 60)
    .translate((0, 0, -twist_offset))
)

# Connect the beams with clever overlaps
connector1 = (cq.Workplane("XY")
    .center(-triangle_size/2, -triangle_size/(2*math.sqrt(3)))
    .box(beam_width * 1.5, beam_width * 1.5, beam_height + twist_offset*2)
)

connector2 = (cq.Workplane("XY")
    .center(triangle_size/2, -triangle_size/(2*math.sqrt(3)))
    .box(beam_width * 1.5, beam_width * 1.5, beam_height + twist_offset*2)
)

connector3 = (cq.Workplane("XY")
    .center(0, triangle_size/math.sqrt(3))
    .box(beam_width * 1.5, beam_width * 1.5, beam_height + twist_offset*2)
)

# Combine all parts
model = beam1.union(beam2).union(beam3)
model = model.union(connector1).union(connector2).union(connector3)

# Add pen holes in the top surfaces
for i in range(3):
    angle = i * 120
    hole_x = hole_spacing * math.cos(math.radians(angle))
    hole_y = hole_spacing * math.sin(math.radians(angle))
    
    pen_hole = (cq.Workplane("XY")
        .center(hole_x, hole_y)
        .circle(hole_diameter/2)
        .extrude(beam_height * 3, both=True)
    )
    model = model.cut(pen_hole)

# Add a subtle twist cut to enhance the illusion
illusion_cut = (cq.Workplane("XZ")
    .center(0, 0)
    .box(triangle_size * 0.3, twist_offset * 0.7, beam_width * 2)
    .rotate((0, 0, 0), (0, 1, 0), 30)
)
model = model.cut(illusion_cut)

show(model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [45]:
#32
import cadquery as cq
from jupyter_cadquery import show
import math

# Commercial Door Pull Handle — two wall plates with countersunk screws, standoffs, and a round pull bar
# Features: unions/cuts, radial (trig) hole arrays, fillets/chamfers, symmetry, iterative loops

# -----------------------
# Parameters (edit freely)
# -----------------------
plate_diameter       = 60.0    # Wall plate outer diameter
plate_thickness      = 6.0     # Wall plate thickness
plate_fillet         = 1.2     # Rim fillet on plates

screw_count          = 4       # Number of mounting screws per plate
screw_circle_diam    = 38.0    # Bolt circle diameter on plates
screw_through_d      = 5.0     # Through hole diameter
screw_csk_d          = 10.0    # Countersink diameter
screw_csk_angle      = 90.0    # Countersink angle (deg)

standoff_length      = 35.0    # Distance from wall plate face to bar centerline plane
standoff_diameter    = 16.0    # Standoff diameter
standoff_count       = 2       # Number of standoffs per plate (top/bottom)

bar_length           = 400.0   # Overall handle length (end-to-end between bends)
bar_diameter         = 28.0    # Handle tube diameter
bar_end_gap          = 80.0    # Distance from each end to nearest standoff center
bar_offset_from_wall = 45.0    # Bar centerline offset from wall (must be > standoff_length/2)

end_cap_thickness    = 3.0     # Thin decorative end caps on bar
edge_chamfer         = 0.6     # Small chamfer on exposed edges

# -----------------------
# Derived values
# -----------------------
plate_r   = plate_diameter / 2
screw_r   = screw_circle_diam / 2
bar_r     = bar_diameter / 2
standoff_r= standoff_diameter / 2

# Sanity: keep bar clear of wall given standoff length
bar_center_z = standoff_length + bar_offset_from_wall

# Standoff vertical positions on plate (symmetric)
standoff_pitch = plate_r * 0.55
standoff_y_positions = (-standoff_pitch, standoff_pitch)

# -----------------------
# Build a single wall plate with countersunk screw holes
# -----------------------
plate = cq.Workplane("XY").circle(plate_r).extrude(plate_thickness)
# Fillet plate rims (top/bottom circular edges)
plate = plate.faces(">Z or <Z").edges("%Circle").fillet(min(plate_fillet, plate_thickness/2 - 0.01))

# Screw holes placed radially using trig
screw_pts = [(screw_r*math.cos(2*math.pi*i/screw_count),
              screw_r*math.sin(2*math.pi*i/screw_count)) for i in range(screw_count)]
plate = (
    plate.faces(">Z").workplane(centerOption="CenterOfMass")
    .pushPoints(screw_pts)
    .cskHole(screw_through_d, screw_csk_d, screw_csk_angle)
)

# -----------------------
# Add two standoffs to the plate (unions)
# -----------------------
standoff_unit = cq.Workplane("XY").circle(standoff_r).extrude(standoff_length)
for y in standoff_y_positions:
    plate = plate.union(standoff_unit.translate((0, y, plate_thickness)))

# -----------------------
# Duplicate plate to the other end along X
# -----------------------
plate_spacing = bar_length
left_plate  = plate.translate((-plate_spacing/2, 0, 0))
right_plate = plate.translate(( plate_spacing/2, 0, 0))

# -----------------------
# Pull bar: straight tube with thin end caps (unions)
# -----------------------
bar_core = (
    cq.Workplane("YZ")
    .center(0, bar_center_z)         # (Y,Z) plane; X will be length direction
    .circle(bar_r)
    .extrude(bar_length, both=True)  # symmetric along +X/-X
)

# Decorative thin end caps at both ends
cap = (
    cq.Workplane("YZ")
    .center(0, bar_center_z)
    .circle(bar_r)
    .extrude(end_cap_thickness)
)
bar = (
    bar_core
    .union(cap.translate(( bar_length, 0, 0)))
    .union(cap.translate((-bar_length, 0, 0)))
)

# -----------------------
# Bar-to-standoff joiners (short collars) and clearance bores
# -----------------------
# Collars: small coaxial rings to visually bridge standoff to bar (union)
collar_thick = 3.0
collar = (
    cq.Workplane("YZ")
    .center(0, bar_center_z)
    .circle(max(bar_r, standoff_r)*1.02)   # tiny oversize for a smooth visual blend
    .extrude(collar_thick)
)

# Place collars at standoff Y positions and both plate X positions
for sx in (-1, 1):
    x_base = sx * (plate_spacing/2)
    for y in standoff_y_positions:
        bar = bar.union(collar.translate((x_base, y, 0)))

# Clearance bore through each standoff to receive bar (cut)
# (Drill a hole coaxial with bar centerline through the standoff depth)
bore = (
    cq.Workplane("YZ")
    .center(0, bar_center_z)
    .circle(bar_r*0.98)    # slight clearance
    .extrude(standoff_length + collar_thick + 1.0)   # ensure full cut depth
)
left_plate  = left_plate.cut(bore.translate((0, 0, 0)))
right_plate = right_plate.cut(bore.translate((0, 0, 0)))

# -----------------------
# Small edge chamfers on exposed ends (visual de-burr)
# -----------------------
bar = bar.faces(">X or <X").edges("%Circle").chamfer(edge_chamfer)

# -----------------------
# Assemble complete handle
# -----------------------
model = left_plate.union(right_plate).union(bar)

# -----------------------
# Display
# -----------------------
show(model)


+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [50]:
#33
import cadquery as cq
from jupyter_cadquery import show
import math

# Commercial Wall Sign Plaque — rounded rectangular plate with countersunk mounts,
# back standoffs, and a centered grid of decorative slots (NO arc holes)

# -----------------------
# Parameters (edit freely)
# -----------------------
plate_length       = 200.0   # X size of plaque
plate_width        = 120.0   # Y size of plaque
plate_thickness    = 4.0     # Z thickness of plaque
corner_radius      = 8.0     # Vertical corner soften (safe fillet)

mount_inset_x      = 18.0    # Inset of mounting holes from left/right edges
mount_inset_y      = 18.0    # Inset of mounting holes from top/bottom edges
mount_hole_d       = 5.0     # Through hole diameter
mount_csk_d        = 10.0    # Countersink diameter
mount_csk_angle    = 90.0    # Countersink angle (deg)

boss_diameter      = 14.0    # Back standoff boss diameter (union on rear)
boss_height        = 8.0     # Boss height (sticks out behind plate)
boss_fillet        = 0.8     # Small fillet on boss rims

slot_rows          = 3       # Decorative vent slots across center band
slot_cols          = 7
slot_length        = 18.0
slot_width         = 4.0
slot_pitch_x       = 24.0
slot_pitch_y       = 14.0

# -----------------------
# Base plaque: box extruded up from Z=0; soften vertical corners
# -----------------------
safe_corner = min(corner_radius, plate_thickness/2 - 0.01, min(plate_length, plate_width)/6)
plaque = (
    cq.Workplane("XY")
    .box(plate_length, plate_width, plate_thickness, centered=(True, True, False))
    .edges("|Z").fillet(max(0.2, safe_corner))  # robust vertical-edge fillet
)

# -----------------------
# Four countersunk mounting holes (practical feature)
# -----------------------
mount_pts = [
    ( sx*(plate_length/2 - mount_inset_x), sy*(plate_width/2 - mount_inset_y) )
    for sx in (-1, 1) for sy in (-1, 1)
]
plaque = (
    plaque.faces(">Z").workplane(centerOption="CenterOfMass")
    .pushPoints(mount_pts)
    .cskHole(mount_hole_d, mount_csk_d, mount_csk_angle)
)

# -----------------------
# Back standoff bosses (unions) aligned with mounting holes
# -----------------------
boss_r = boss_diameter/2
boss_unit = cq.Workplane("XY").circle(boss_r).extrude(boss_height)
for (x, y) in mount_pts:
    plaque = plaque.union(boss_unit.translate((x, y, -boss_height)))

# Soften boss rims safely (circular edges only)
plaque = plaque.faces("<Z").edges("%Circle").fillet(min(boss_fillet, boss_height/2 - 0.01))

# -----------------------
# Decorative slots (robust: workplane + slot2D + cutThruAll)
# -----------------------
start_x = -(slot_cols-1)*slot_pitch_x/2
start_y = -(slot_rows-1)*slot_pitch_y/2
for r in range(slot_rows):
    for c in range(slot_cols):
        px = start_x + c*slot_pitch_x
        py = start_y + r*slot_pitch_y
        plaque = (
            plaque
            .faces(">Z").workplane(centerOption="CenterOfMass")
            .center(px, py)
            .slot2D(slot_length, slot_width)
            .cutThruAll()
        )

# -----------------------
# Final display (no arc holes)
# -----------------------
model = plaque
show(model)


+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [54]:
#34
import cadquery as cq
from jupyter_cadquery import show
import math  # not strictly needed

# Simple 90° Corner Bracket — two plates with countersunk mounting holes

# -----------------------
# Parameters (edit freely)
# -----------------------
leg_x           = 80.0   # Length along X for each leg
leg_y           = 30.0   # Width along Y (common for both legs)
plate_thickness = 4.0    # Plate thickness
wall_height     = 60.0   # Vertical leg height (Z)

hole_d          = 5.0    # Through hole diameter
csk_diameter    = 10.0   # Countersink diameter
csk_angle       = 90.0   # Countersink angle in degrees
hole_edge_inset = 15.0   # Inset of holes from edges
holes_per_leg   = 2      # Holes per leg (evenly spaced)

# -----------------------
# Build the two legs (simple boxes); one horizontal, one vertical
# -----------------------
# Horizontal leg: sits on XY, thickness extruded in +Z
h_leg = (
    cq.Workplane("XY")
    .box(leg_x, leg_y, plate_thickness, centered=(True, True, False))
)

# Vertical leg: stands up from back edge of horizontal leg
v_leg = (
    cq.Workplane("XY")
    .box(leg_x, plate_thickness, wall_height + plate_thickness, centered=(True, True, False))
    .translate((0, leg_y/2 - plate_thickness/2, 0))
)

bracket = h_leg.union(v_leg)

# -----------------------
# Mounting holes on horizontal leg (top face)
# -----------------------
# Evenly spaced along X, centered in Y
if holes_per_leg > 0:
    pitch_h = (leg_x - 2*hole_edge_inset) / (max(holes_per_leg-1, 1))
    x_positions = [(-leg_x/2 + hole_edge_inset + i*pitch_h) for i in range(holes_per_leg)]
    bracket = (
        bracket
        .faces(">Z").workplane(centerOption="CenterOfMass")
        .pushPoints([(x, 0) for x in x_positions])
        .cskHole(hole_d, csk_diameter, csk_angle)
    )

# -----------------------
# Mounting holes on vertical leg (outer face)
# -----------------------
# Place holes at mid-height along Z, spaced along X the same way
mid_z = wall_height * 0.5
bracket = (
    bracket
    .faces(">Y").workplane(centerOption="CenterOfMass")
    .center(0, mid_z)
    .pushPoints([(x, 0) for x in x_positions])
    .cskHole(hole_d, csk_diameter, csk_angle)
)

# -----------------------
# Final display
# -----------------------
model = bracket
show(model)


+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [1]:
#35
import cadquery as cq
from jupyter_cadquery import show
import math

# Simple Cable Organizer Bar — slim bar with semicircular cable channels, entry slots, and mounting holes

# -----------------------
# Parameters (edit freely)
# -----------------------
bar_length        = 140.0   # X size of the bar
bar_width         = 28.0    # Y size
bar_height        = 12.0    # Z height

channel_count     = 5       # Number of top cable channels
channel_radius    = 4.5     # Radius of each semicircular channel
channel_spacing   = 24.0    # Spacing along X between channel centers
channel_offset_y  = 0.0     # Lateral offset (0 = centered)

entry_slot_w      = 6.0     # Side entry slot width (Y direction)
entry_slot_h      = 6.0     # Slot height (Z direction)
entry_inset_x     = 12.0    # Inset from bar ends along X

mount_hole_d      = 4.0     # Through hole diameter (for screws)
csk_diameter      = 8.0     # Countersink diameter
csk_angle         = 90.0    # Countersink angle in degrees
mount_inset_x     = 18.0    # Inset from ends for mounting holes

tape_pocket_h     = 1.5     # Shallow underside pocket for adhesive tape
tape_margin       = 2.0     # Margin from outer edges around the tape pocket

# -----------------------
# Base bar
# -----------------------
bar = cq.Workplane("XY").box(bar_length, bar_width, bar_height, centered=(True, True, False))

# -----------------------
# Top semicircular cable channels (robust cylindrical cuts)
# -----------------------
first_x = - (channel_spacing * (channel_count-1) / 2.0)
cutter_len = bar_width + 2.0
cyl = cq.Workplane("YZ").center(channel_offset_y, bar_height).circle(channel_radius).extrude(cutter_len, both=True)
for i in range(channel_count):
    x = first_x + i*channel_spacing
    bar = bar.cut(cyl.translate((x, 0, 0)))

# -----------------------
# Side entry slots (rectangular cuts from each side near the ends)
# -----------------------
slot_len = entry_slot_w
slot = (
    cq.Workplane("XZ")
    .center(0, bar_height - entry_slot_h/2)
    .rect(entry_slot_h, entry_slot_w)  # (Z,X) plane dims, will extrude along Y
    .extrude(bar_width/2 + 1.0)        # cut from one side
)
# Left end slot (from +Y) and mirrored to -Y side
bar = bar.cut(slot.translate((-bar_length/2 + entry_inset_x, 0, 0)))
bar = bar.cut(slot.rotate((0,0,0),(1,0,0),180).translate((-bar_length/2 + entry_inset_x, 0, 0)))
# Right end slots
bar = bar.cut(slot.translate(( bar_length/2 - entry_inset_x, 0, 0)))
bar = bar.cut(slot.rotate((0,0,0),(1,0,0),180).translate(( bar_length/2 - entry_inset_x, 0, 0)))

# -----------------------
# Mounting holes (countersunk) on top face
# -----------------------
mount_pts = [(-bar_length/2 + mount_inset_x, 0), (bar_length/2 - mount_inset_x, 0)]
bar = (
    bar.faces(">Z").workplane(centerOption="CenterOfMass")
    .pushPoints(mount_pts)
    .cskHole(mount_hole_d, csk_diameter, csk_angle)
)

# -----------------------
# Underside tape pocket (shallow rectangular recess)
# -----------------------
pocket_len = bar_length - 2*tape_margin
pocket_wid = bar_width  - 2*tape_margin
pocket = cq.Workplane("XY").rect(pocket_len, pocket_wid).extrude(tape_pocket_h)
bar = bar.cut(pocket.translate((0, 0, 0)))  # cut downward from the bottom

# -----------------------
# Final display
# -----------------------
model = bar
show(model)


Overwriting auto display for cadquery Workplane and Shape
+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [14]:
#36
import cadquery as cq

# Parameters
width = 80
height = 60
base_depth = 40
thickness = 5
hole_diameter = 6
hole_padding = 12

# Create the base plate, with its origin at the bottom center
bracket = cq.Workplane("XY").box(width, base_depth, thickness, centered=(True, True, False))

# Add the vertical flange to the back face of the base
# A new workplane is created on the face selected by .faces("<Y")
bracket = (
    bracket.faces("<Y")
    .workplane()
    .rect(width, height, centered=(True, False))
    .extrude(thickness)
)

# Add holes to the base plate
bracket = (
    bracket.faces(">Z")
    .workplane()
    .rect(width - 2 * hole_padding, base_depth - 2 * hole_padding)
    .vertices()
    .hole(hole_diameter)
)

# Add holes to the vertical plate
bracket = (
    bracket.faces("<Y")
    .workplane()
    .rect(width - 2 * hole_padding, height - 2 * hole_padding)
    .vertices()
    .hole(hole_diameter)
)

# Display the final bracket
show(bracket)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [15]:
#37
import cadquery as cq

# Enclosure dimensions
length = 100
width = 70
height = 35
wall_thickness = 2.5
lid_height = 15

# Create the base of the enclosure
base = (
    cq.Workplane("XY")
    .box(length, width, height)
    .faces(">Z")
    .shell(-wall_thickness) # Negative shell hollows the inside
)

# Create a matching lid with a slight inset for a better fit
inset = wall_thickness / 2.0
lid = (
    cq.Workplane("XY")
    .box(length - inset, width - inset, lid_height)
    .faces("<Z")
    .shell(wall_thickness) # Positive shell creates a lip on the outside
    .translate((0, 0, (height - wall_thickness)/2.0))
)

# Display both parts of the enclosure together
show(base, lid)

c+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [17]:
#38
import cadquery as cq

# --- Parameters ---
base_length = 60.0
base_width = 40.0
base_height = 5.0

fin_height = 15.0
fin_thickness = 1.5
fin_count = 15

hole_diameter = 3.0
hole_padding = 5.0 # Distance from edge to hole center

# --- Model Creation ---

# 1. Create the solid base plate
heat_sink = cq.Workplane("XY").box(base_length, base_width, base_height)

# 2. Create an array of fins on the top face
# We use rarray (rectangular array) to pattern a simple rectangle sketch
# which is then extruded.
fins = (
    heat_sink.faces(">Z")
    .workplane()
    .rarray(
        xSpacing=base_length / (fin_count -1), # Calculated spacing
        ySpacing=1,                          # Not used, only 1 row
        xCount=fin_count,
        yCount=1,
        center=True
    )
    .rect(fin_thickness, base_width * 0.9) # Each fin is 90% of the base width
    .extrude(fin_height)
)

# The 'fins' object now contains both the base and the fins

# 3. Add mounting holes to the corners of the base
# We select the top face again to define the hole locations
result = (
    fins.faces(">Z")
    .workplane()
    .rect(
        base_length - 2 * hole_padding,
        base_width - 2 * hole_padding,
        forConstruction=True # The rectangle is a guide, not part of the solid
    )
    .vertices() # Select the 4 corners of the guide rectangle
    .hole(hole_diameter)
)

# --- Display the Result ---
show(result)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [22]:
#39
import cadquery as cq
from cadquery.selectors import RadiusNthSelector

# --- Parameters ---
outer_diameter = 60.0
wall_thickness = 4.0
pipe_length = 20.0

flange_diameter = 120.0
flange_thickness = 15.0

bolt_hole_count = 6
bolt_hole_diameter = 10.0
bolt_circle_diameter = 90.0 # The diameter of the circle on which holes lie

# --- Model Creation ---

# 1. Create the main pipe section
pipe = (
    cq.Workplane("XY")
    .circle(outer_diameter / 2.0)
    .circle((outer_diameter / 2.0) - wall_thickness)
    .extrude(pipe_length)
)

# 2. Add the flange to the end of the pipe
flange = (
    pipe.faces(">Z")
    .workplane()
    .circle(flange_diameter / 2.0)
    .extrude(flange_thickness)
)

# 3. Add the bolt holes to the flange face
result = (
    flange.faces(">Z")
    .workplane()
    .polarArray(
        radius=bolt_circle_diameter / 2.0,
        startAngle=0,
        angle=360,
        count=bolt_hole_count
    )
    .hole(bolt_hole_diameter)
)

# 4. Correctly select and chamfer the flange's outer edge.
# This chain selects the top face, then filters its edges for the one
# with the largest radius (the 0th one), and applies the chamfer.
result = result.faces(">Z").edges(RadiusNthSelector(0)).chamfer(2.0)

# --- Display the Result ---
show(result)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [32]:
#40
import cadquery as cq

# --- Parameters ---
length = 250.0
height = 150.0
depth = 140.0
thickness = 10.0

divider_count = 4
fillet_radius = 5.0 # Radius for rounded internal corners

# --- Model Creation ---

# 1. Create the base plate and walls as separate solids first
base = cq.Workplane("XY").box(length, depth, thickness, centered=(True, True, False))

back_wall = base.faces("<Y").workplane().rect(length, height, centered=(True, False)).extrude(thickness)
left_wall = base.faces("<X").workplane().rect(depth, height, centered=(True, False)).extrude(thickness)
right_wall = base.faces(">X").workplane().rect(depth, height, centered=(True, False)).extrude(-thickness)

# 2. Union the base and walls to form the main "U" shape
tray = base.union(back_wall).union(left_wall).union(right_wall)

# 3. Perform a robust fillet on the three main internal corners
# We select the three internal vertical faces and find the edges they share.
tray_filleted = (
    tray.faces("<Y[-1] or <X[-1] or >X[-1]") # Select the 3 inner vertical faces
    .edges("%Line") # Select all straight edges on these faces
    .fillet(fillet_radius)
)

# 4. Create the dividers as a completely separate body
dividers = (
    cq.Workplane("XY", origin=(0, 0, thickness)) # Workplane is on top of the base
    .rarray(
        xSpacing=(length - thickness) / (divider_count + 1),
        ySpacing=1,
        xCount=divider_count,
        yCount=1,
        center=True,
    )
    .rect(thickness, depth - thickness) # Ensure dividers fit inside back wall
    .extrude(height - thickness)
)

# 5. Union the filleted tray with the dividers to get the final result
result = tray_filleted.union(dividers)

# --- Display the Result ---
show(result)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [51]:
#41
import cadquery as cq
from jupyter_cadquery import show

# SIMPLE SPOON - no decorations

# Create handle as a long rectangular bar
handle = (cq.Workplane("XY")
    .box(100, 8, 2)  # Long, narrow, flat
    .translate((50, 0, 0))  # Position it
)

# Create spoon bowl - simple oval with depression
bowl = (cq.Workplane("XY")
    .ellipse(18, 25)
    .extrude(5)
)

# Scoop out the bowl
scoop = (cq.Workplane("XY")
    .workplane(offset=1.5)
    .ellipse(15, 22)
    .extrude(5)
)
bowl = bowl.cut(scoop)

# Combine handle and bowl
model = handle.union(bowl)

show(model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [1]:
#42
import cadquery as cq
from jupyter_cadquery import show

# Simple Shoe Organizer - Basic 3-tier rack

# Parameters
rack_width = 80.0
rack_depth = 30.0
shelf_thickness = 3.0
tier_height = 20.0
num_tiers = 3

# Side supports
left_support = (cq.Workplane("YZ")
    .box(rack_depth, num_tiers * tier_height, shelf_thickness)
    .translate((-rack_width/2, 0, num_tiers * tier_height/2))
)

right_support = (cq.Workplane("YZ")
    .box(rack_depth, num_tiers * tier_height, shelf_thickness)
    .translate((rack_width/2, 0, num_tiers * tier_height/2))
)

model = left_support.union(right_support)

# Add shelves at different heights
for i in range(num_tiers):
    shelf_height = i * tier_height + shelf_thickness/2
    
    # Create slanted shelf for better shoe display
    shelf = (cq.Workplane("XY")
        .workplane(offset=shelf_height)
        .box(rack_width, rack_depth, shelf_thickness)
    )
    
    # Add small lip at front to prevent shoes sliding
    if i > 0:  # Not on bottom shelf
        lip = (cq.Workplane("XY")
            .workplane(offset=shelf_height)
            .center(0, -rack_depth/2 + 1)
            .box(rack_width - 6, 2, 5)
        )
        shelf = shelf.union(lip)
    
    model = model.union(shelf)

# Add ventilation holes in shelves
for tier in range(1, num_tiers):
    for x in range(-2, 3):
        hole = (cq.Workplane("XY")
            .workplane(offset=tier * tier_height)
            .center(x * 15, 0)
            .circle(4)
            .extrude(shelf_thickness + 1)
        )
        model = model.cut(hole)

show(model)

Overwriting auto display for cadquery Workplane and Shape
+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [2]:
#43
import cadquery as cq
from jupyter_cadquery import show
import math

# Tree-Style Towel/Clothes Dryer - SIMPLIFIED

# Parameters
trunk_height = 120.0
trunk_diameter = 8.0
base_diameter = 40.0
base_height = 5.0

# Branch parameters
branch_length = 35.0
branch_diameter = 5.0
num_levels = 3
branches_per_level = 3

# Create stable base (roots) - simple cylinder
base = (cq.Workplane("XY")
    .circle(base_diameter/2)
    .extrude(base_height)
)

# Create main trunk
trunk = (cq.Workplane("XY")
    .workplane(offset=base_height)
    .circle(trunk_diameter/2)
    .extrude(trunk_height)
)

model = base.union(trunk)

# Add branches - simpler approach
for level in range(num_levels):
    height = base_height + 40 + level * 35
    
    for i in range(branches_per_level):
        angle = (360 / branches_per_level) * i + (level * 30)
        
        # Simple straight branch
        branch_x = branch_length * math.cos(math.radians(angle))
        branch_y = branch_length * math.sin(math.radians(angle))
        
        # Create branch as simple box rotated to position
        branch = (cq.Workplane("XY")
            .workplane(offset=height)
            .center(branch_x/2, branch_y/2)
            .rect(branch_length, branch_diameter)
            .extrude(branch_diameter)
            .rotate((0, 0, height), (0, 0, 1), angle)
        )
        
        model = model.union(branch)

# Add simple cylinder on top instead of sphere
top = (cq.Workplane("XY")
    .workplane(offset=trunk_height + base_height)
    .circle(6)
    .extrude(4)
)
model = model.union(top)

show(model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [3]:
#44
import cadquery as cq
from jupyter_cadquery import show

# Accessory Organizer - Desktop jewelry/watch/keys organizer

# Parameters
base_length = 120.0
base_width = 80.0
base_thickness = 5.0

# Compartment parameters
wall_thickness = 2.0
compartment_depth = 15.0

# Ring holder parameters
ring_post_diameter = 12.0
ring_post_height = 40.0

# Create base tray
base = (cq.Workplane("XY")
    .box(base_length, base_width, base_thickness)
)

# Create outer walls
walls = (cq.Workplane("XY")
    .box(base_length, base_width, compartment_depth)
    .faces(">Z")
    .shell(-wall_thickness)
)

model = base.union(walls)

# Add divider walls for compartments
# Long divider
divider1 = (cq.Workplane("XZ")
    .workplane(offset=base_width/3)
    .center(-base_length/4, base_thickness + compartment_depth/2)
    .box(base_length/2, compartment_depth, wall_thickness)
)

# Short dividers
divider2 = (cq.Workplane("YZ")
    .workplane(offset=0)
    .center(-base_width/6, base_thickness + compartment_depth/2)
    .box(base_width/3, compartment_depth, wall_thickness)
)

divider3 = (cq.Workplane("YZ")
    .workplane(offset=-base_length/4)
    .center(-base_width/6, base_thickness + compartment_depth/2)
    .box(base_width/3, compartment_depth, wall_thickness)
)

model = model.union(divider1).union(divider2).union(divider3)

# Add ring/bracelet holder post
ring_post = (cq.Workplane("XY")
    .workplane(offset=base_thickness)
    .center(base_length/4, 0)
    .circle(ring_post_diameter/2)
    .extrude(ring_post_height)
)

# Taper the top of the post
post_top = (cq.Workplane("XY")
    .workplane(offset=base_thickness + ring_post_height)
    .center(base_length/4, 0)
    .circle(ring_post_diameter/2)
    .workplane(offset=5)
    .circle(ring_post_diameter/3)
    .loft()
)

model = model.union(ring_post).union(post_top)

# Add watch rest groove
watch_groove = (cq.Workplane("XY")
    .workplane(offset=base_thickness)
    .center(base_length/4, -base_width/3)
    .box(30, 8, 5)
)
model = model.cut(watch_groove)

# Add small holes for earrings
for i in range(4):
    for j in range(2):
        hole = (cq.Workplane("XY")
            .center(-base_length/3 + i*8, base_width/3 - j*8)
            .circle(1)
            .extrude(base_thickness + 1)
        )
        model = model.cut(hole)

show(model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [6]:
#45
import cadquery as cq
from jupyter_cadquery import show

# Umbrella Stand/Hanger - Simple and functional

# Parameters
base_diameter = 80.0
base_height = 8.0
pole_height = 100.0
pole_diameter = 30.0
hook_radius = 15.0
num_hooks = 4

# Drip tray parameters
tray_diameter = 70.0
tray_depth = 5.0

# Create weighted base
base = (cq.Workplane("XY")
    .circle(base_diameter/2)
    .extrude(base_height)
)

# Create main cylinder with hollow center for umbrellas
main_holder = (cq.Workplane("XY")
    .workplane(offset=base_height)
    .circle(pole_diameter/2)
    .circle(pole_diameter/2 - 3)
    .extrude(pole_height)
)

model = base.union(main_holder)

# Add hooks around the top for hanging umbrellas
for i in range(num_hooks):
    angle = 360 * i / num_hooks
    
    # Hook arm extending outward
    hook_arm = (cq.Workplane("XZ")
        .workplane(offset=0)
        .center(0, base_height + pole_height - 10)
        .circle(3)
        .extrude(hook_radius + 5)
        .rotate((0, 0, 0), (0, 0, 1), angle)
    )
    
    # Hook tip pointing up
    hook_x = (hook_radius + 5) * math.cos(math.radians(angle))
    hook_y = (hook_radius + 5) * math.sin(math.radians(angle))
    
    hook_tip = (cq.Workplane("XY")
        .workplane(offset=base_height + pole_height - 10)
        .center(hook_x, hook_y)
        .circle(3)
        .extrude(12)
    )
    
    model = model.union(hook_arm).union(hook_tip)

# Add drip tray indentation in base
drip_tray = (cq.Workplane("XY")
    .workplane(offset=tray_depth)
    .circle(tray_diameter/2)
    .circle(pole_diameter/2 + 2)
    .extrude(tray_depth)
)
model = model.cut(drip_tray)

# Add drainage holes in tray
for i in range(6):
    angle = 60 * i
    hole_x = (tray_diameter/2 - 10) * math.cos(math.radians(angle))
    hole_y = (tray_diameter/2 - 10) * math.sin(math.radians(angle))
    
    drain = (cq.Workplane("XY")
        .center(hole_x, hole_y)
        .circle(2)
        .extrude(base_height)
    )
    model = model.cut(drain)

import math
show(model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [7]:
#46
import cadquery as cq
from jupyter_cadquery import show

# Dish Organizer/Drying Rack - Compact countertop design

# Parameters
rack_length = 120.0
rack_width = 80.0
rack_height = 40.0
base_thickness = 3.0

# Plate slots parameters
slot_width = 3.0
slot_spacing = 15.0
num_slots = 6

# Create base with drainage slope
base = (cq.Workplane("XY")
    .box(rack_length, rack_width, base_thickness)
)

# Add raised edges to contain water
edge_front = (cq.Workplane("XY")
    .center(0, -rack_width/2 + 2)
    .box(rack_length, 4, 8)
)

edge_back = (cq.Workplane("XY")
    .center(0, rack_width/2 - 2)
    .box(rack_length, 4, 8)
)

edge_left = (cq.Workplane("XY")
    .center(-rack_length/2 + 2, 0)
    .box(4, rack_width, 8)
)

edge_right = (cq.Workplane("XY")
    .center(rack_length/2 - 2, 0)
    .box(4, rack_width - 20, 8)  # Gap for drainage
)

model = base.union(edge_front).union(edge_back).union(edge_left).union(edge_right)

# Add plate dividers
for i in range(num_slots):
    x_pos = -rack_length/2 + 20 + i * slot_spacing
    
    divider = (cq.Workplane("XZ")
        .workplane(offset=x_pos)
        .center(0, rack_height/2)
        .box(rack_width - 10, rack_height, slot_width)
    )
    model = model.union(divider)

# Add cup/mug pegs on one side
for j in range(4):
    peg_y = -rack_width/2 + 15 + j * 15
    
    peg = (cq.Workplane("XY")
        .workplane(offset=base_thickness)
        .center(rack_length/2 - 20, peg_y)
        .circle(3)
        .extrude(25)
    )
    model = model.union(peg)

# Add drainage grooves
for k in range(5):
    groove = (cq.Workplane("XY")
        .center(-30 + k * 15, 0)
        .box(2, rack_width - 8, 1)
    )
    model = model.cut(groove)

# Add utensil holder section
utensil_box = (cq.Workplane("XY")
    .workplane(offset=base_thickness)
    .center(-rack_length/2 + 10, rack_width/2 - 15)
    .box(15, 20, 30)
    .faces(">Z")
    .shell(-2)
)
model = model.union(utensil_box)

show(model)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [12]:
#47
import cadquery as cq
from jupyter_cadquery import show

# Industrial Pipe Fitting Organizer - Workshop storage

# Parameters
plate_width = 150.0
plate_height = 100.0
plate_thickness = 10.0

# Peg parameters
peg_diameter = 8.0
peg_length = 40.0
peg_spacing = 30.0

# Mounting hole parameters
mount_hole_dia = 6.0
mount_spacing = 120.0

# --- Backplate: bottom on XY plane, thickness along +Z ---
backplate = (
    cq.Workplane("XY")
    .box(plate_width, plate_height, plate_thickness, centered=(True, True, False))
    .faces(">Z").edges().chamfer(2)
)

# --- Mounting holes + counterbores ---
for x in [-mount_spacing/2, mount_spacing/2]:
    for y in [-35, 35]:
        # through-hole starting at bottom, extruding through plate
        mount_hole = (
            cq.Workplane("XY")
            .center(x, y)
            .circle(mount_hole_dia/2)
            .extrude(plate_thickness + 1)
        )
        backplate = backplate.cut(mount_hole)

        # counterbore from front face downward
        counterbore = (
            cq.Workplane("XY")
            .workplane(offset=plate_thickness)  # front face at Z = plate_thickness
            .center(x, y)
            .circle(mount_hole_dia)
            .extrude(-4)                       # cut into the plate
        )
        backplate = backplate.cut(counterbore)

# --- Storage pegs on front face ---
for row in range(3):
    for col in range(4):
        x_pos = -45 + col * peg_spacing
        y_pos = -20 + row * peg_spacing

        # tapered peg starting from front face
        peg = (
            cq.Workplane("XY")
            .workplane(offset=plate_thickness)   # front face
            .center(x_pos, y_pos)
            .circle(peg_diameter/2)
            .workplane(offset=peg_length - 5)
            .circle(peg_diameter/2 - 1)
            .loft()
        )
        backplate = backplate.union(peg)

        # retention ball at tip
        ball = (
            cq.Workplane("XY")
            .workplane(offset=plate_thickness + peg_length - 3)
            .center(x_pos, y_pos)
            .sphere(peg_diameter/2 + 0.5)
        )
        backplate = backplate.union(ball)

# --- Reinforcement ribs (gussets) touching the plate ---
for i in range(2):
    rib = (
        cq.Workplane("XZ")
        .workplane(offset=-30 + i * 60)   # along plate height (Y)
        # origin at (0,0) => Z=0 is plate bottom, X across width
        .polyline([
            (-plate_width/2 + 10, 0),   # bottom
            (-plate_width/2 + 10, 3),   # up a bit on plate
            (-plate_width/2 + 13, 0)
        ])
        .close()
        .extrude(5)                      # along +Y
    )
    backplate = backplate.union(rib)

# --- Label recess, cut from front face ---
label_area = (
    cq.Workplane("XY")
    .workplane(offset=plate_thickness)           # front face
    .center(0, -plate_height/2 + 10)
    .rect(80, 12)
    .extrude(-1.5)                               # cut into plate
)
backplate = backplate.cut(label_area)

show(backplate)


+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [14]:
#48
import cadquery as cq
from jupyter_cadquery import show

# =========================
# Wall-Mounted Tool Hanger
# =========================

# Backplate parameters
plate_width      = 180.0
plate_height     = 60.0
plate_thickness  = 8.0

# Hook parameters
num_hooks        = 6
hook_spacing     = 25.0
hook_width       = 10.0
hook_thickness   = 6.0
hook_height      = 25.0
hook_depth       = 20.0

# Mounting hole parameters
mount_hole_dia   = 5.0
edge_margin_x    = 20.0
edge_margin_y    = 15.0

# --------------------
# Backplate on XY base
# --------------------
backplate = (
    cq.Workplane("XY")
    .box(plate_width, plate_height, plate_thickness, centered=(True, True, False))
    .faces(">Z").edges().chamfer(1.5)
)

# --------------------
# Mounting holes + counterbore
# --------------------
for x in (-plate_width/2 + edge_margin_x, plate_width/2 - edge_margin_x):
    for y in (-plate_height/2 + edge_margin_y, plate_height/2 - edge_margin_y):

        # Through hole from bottom
        through = (
            cq.Workplane("XY")
            .center(x, y)
            .circle(mount_hole_dia / 2)
            .extrude(plate_thickness + 1)
        )
        backplate = backplate.cut(through)

        # Counterbore from the front face (Z = plate_thickness)
        cbore = (
            cq.Workplane("XY")
            .workplane(offset=plate_thickness)
            .center(x, y)
            .circle(mount_hole_dia)
            .extrude(-3.0)
        )
        backplate = backplate.cut(cbore)

# --------------------
# Hooks array
# --------------------
hook_row_width = (num_hooks - 1) * hook_spacing
start_x = -hook_row_width / 2

for i in range(num_hooks):
    x_pos = start_x + i * hook_spacing
    y_pos = 0  # centered vertically

    # Base pad on plate face
    base_pad = (
        cq.Workplane("XY")
        .workplane(offset=plate_thickness)
        .center(x_pos, y_pos)
        .rect(hook_width, hook_thickness)
        .extrude(4.0)
    )

    # Vertical part of the hook
    vertical = (
        base_pad.faces(">Z")
        .workplane(centerOption="CenterOfMass")
        .rect(hook_width, hook_thickness)
        .extrude(hook_height)
    )

    # Forward projection
    forward = (
        vertical.faces(">Z")
        .workplane(centerOption="CenterOfMass")
        .transformed(rotate=(90, 0, 0))
        .rect(hook_width, hook_thickness)
        .extrude(hook_depth)
    )

    # Add smooth fillet for industrial look
    hook = forward.edges("|Z").fillet(1.0)

    backplate = backplate.union(hook)

# --------------------
# Label recess
# --------------------
label = (
    cq.Workplane("XY")
    .workplane(offset=plate_thickness)
    .center(0, -plate_height/2 + 12)
    .rect(80, 10)
    .extrude(-1.0)
)

backplate = backplate.cut(label)

show(backplate)


+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [15]:
#49
import cadquery as cq
from jupyter_cadquery import show

# =========================
# Cable / Hose Cleat Strip
# =========================

# Backplate parameters
plate_width     = 160.0
plate_height    = 50.0
plate_thickness = 8.0

# Cleat parameters
num_cleats      = 5
cleat_spacing   = 30.0        # center-to-center along X
cleat_width     = 16.0        # along X
cleat_height    = 26.0        # along Y on the plate
cleat_depth     = 24.0        # how far out from wall (+Z)

cable_radius    = 8.0         # radius of the cable groove

# Mounting hole parameters
mount_hole_dia  = 5.0
edge_margin_x   = 20.0
edge_margin_y   = 12.0

# --------------------
# Backplate (bottom at Z = 0)
# --------------------
backplate = (
    cq.Workplane("XY")
    .box(plate_width, plate_height, plate_thickness,
         centered=(True, True, False))
    .faces(">Z").edges().chamfer(1.5)
)

# --------------------
# Mounting holes with counterbores
# --------------------
for x in (-plate_width/2 + edge_margin_x, plate_width/2 - edge_margin_x):
    for y in (-plate_height/2 + edge_margin_y, plate_height/2 - edge_margin_y):

        # Through hole from bottom
        through = (
            cq.Workplane("XY")
            .center(x, y)
            .circle(mount_hole_dia / 2)
            .extrude(plate_thickness + 1)
        )
        backplate = backplate.cut(through)

        # Counterbore from front face (Z = plate_thickness)
        cbore = (
            cq.Workplane("XY")
            .workplane(offset=plate_thickness)
            .center(x, y)
            .circle(mount_hole_dia)
            .extrude(-3.0)
        )
        backplate = backplate.cut(cbore)

# --------------------
# Cleats with rounded cable grooves
# --------------------
row_width = (num_cleats - 1) * cleat_spacing
start_x = -row_width / 2
y_pos = 0  # centered vertically on plate

for i in range(num_cleats):
    x_pos = start_x + i * cleat_spacing

    # Rectangular cleat block extruded out of the front face
    cleat_block = (
        cq.Workplane("XY")
        .workplane(offset=plate_thickness)
        .center(x_pos, y_pos)
        .rect(cleat_width, cleat_height)
        .extrude(cleat_depth)
    )

    # Rounded groove for the cable: cylinder through cleat
    groove = (
        cq.Workplane("YZ")
        .workplane(offset=x_pos)
        .center(y_pos, plate_thickness + cleat_depth / 2)
        .circle(cable_radius)
        .extrude(cleat_width + 2)     # a little wider than the cleat
    )

    cleat = cleat_block.cut(groove)

    # Soften edges slightly
    cleat = cleat.edges("|Z").fillet(1.0)

    backplate = backplate.union(cleat)

# --------------------
# Small label recess at bottom
# --------------------
label = (
    cq.Workplane("XY")
    .workplane(offset=plate_thickness)
    .center(0, -plate_height/2 + 10)
    .rect(70, 8)
    .extrude(-1.0)
)
backplate = backplate.cut(label)

show(backplate)


+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

In [20]:
#50
import cadquery as cq
from jupyter_cadquery import show

# =========================
# Simple Industrial Gear
# =========================

# Gear parameters
num_teeth       = 18
gear_thickness  = 8.0

pitch_radius    = 30.0       # approx "pitch circle" radius
tooth_depth     = 4.0        # radial tooth height
tooth_width     = 4.0        # along circumference

# Hub & bore
bore_diameter   = 8.0
hub_diameter    = 18.0
hub_thickness   = 6.0

# --------------------
# Gear blank (disk)
# --------------------
outer_radius = pitch_radius + tooth_depth * 0.5

gear = (
    cq.Workplane("XY")
    .circle(outer_radius)
    .extrude(gear_thickness)
)

# --------------------
# Teeth as rectangular blocks
# --------------------
tooth_radius = pitch_radius + tooth_depth / 2.0

teeth = (
    cq.Workplane("XY")
    .polarArray(tooth_radius, 0, 360, num_teeth)
    # rect(radial, tangential)
    .rect(tooth_depth, tooth_width)
    .extrude(gear_thickness)
)

gear = gear.union(teeth)

# --------------------
# Center bore
# --------------------
bore = (
    cq.Workplane("XY")
    .circle(bore_diameter / 2.0)
    .extrude(gear_thickness + 2.0)
)
gear = gear.cut(bore)

# --------------------
# Hub on one side
# --------------------
hub = (
    cq.Workplane("XY")
    .workplane(offset=gear_thickness)
    .circle(hub_diameter / 2.0)
    .extrude(hub_thickness)
)
gear = gear.union(hub)

# If you want to *try* a fillet and your OCC build is happy, uncomment:
# gear = gear.edges("|Z").fillet(0.8)

show(gear)


+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…